# makable Shorts: the voices

The voices without the rest of the video, made in the run's folder on Drive, where the Shorts notebook (and `python -m shorts`) use them as they are:

1. **The voice**: taken from a recording's sound, such as the intro Gemini Omni filmed, or a video of you. Chatterbox clones it.
2. **The narration** (the `speak` stage): Chatterbox speaks each scene of a script in that voice, `narration/01.wav` and on. Use it to hear a script before making the video, say one written from a recording's steps, and to try Chatterbox's settings (section 2), which then go in the run's `run.json`.
3. **The intro, when the voice came from another recording** (`revoice-intro`): Omni's clip has Omni's voice. Its speech is separated from the background (Demucs) and converted to the cloned voice, keeping the words and timing so the lips still match: `intro/voiced.mp4`.

**Before you start**

- *Runtime → Change runtime type →* **T4 GPU**.
- The recording to take the voice from, on Google Drive. In the voice's cell, `RECORDING` is its path, or a pattern like `…/intro-*.mp4` (the newest match). It's set to the Crumb & Co intro.
- `LANGUAGE` in the setup cell: `"en"` or `"hi"` for Hindi. Hindi is spoken by Chatterbox's multilingual model (its first run downloads it), and its runs get a folder of their own.

## Setup

Chatterbox installs its own torch, older than Colab's. If the session had already loaded Colab's, the install cell restarts it (Colab says the session crashed): go on from the next cell. If Colab asks to restart the session, do that too.

pip lists conflicts with packages Colab already has, such as `google-adk` and `python-fasthtml` wanting a newer `starlette`. That's expected: nothing here uses them. Chatterbox's torch is older than Colab's, so the cell also removes Colab's `torchvision`, which was built for the newer one and can stop `transformers` (which Chatterbox uses) from importing.

In [ ]:
!pip install -q chatterbox-tts demucs
!pip uninstall -q -y torchvision  # built for Colab's newer torch; nothing here uses it

import os
import sys
from importlib.metadata import version

def release(number):  # PyPI's torch says 2.6.0+cu124 when imported and 2.6.0 to pip
    return str(number).split("+")[0]


stale = [name for name in ("torch", "torchaudio", "transformers", "numpy")
         if name in sys.modules and release(sys.modules[name].__version__) != release(version(name))]
if stale:
    print(f"Restarting the session, which has the old {', '.join(stale)} loaded. Then go on from the next cell.", flush=True)
    os.kill(os.getpid(), 9)


In [ ]:
import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

HOME = Path("/content/drive/MyDrive/makable-shorts")
ASSETS = HOME / "assets"
LANGUAGE = "en"  # what the script and narration are in: "en", or "hi" for Hindi
# One folder per video (and language): the Shorts notebook's RUN, to go on there.
RUN = HOME / "runs" / ("crumb-and-co" if LANGUAGE == "en" else f"crumb-and-co-{LANGUAGE}")


In [ ]:
os.makedirs("/content/shorts/stages", exist_ok=True)


In [ ]:
%%writefile /content/shorts/__init__.py
"""makable Shorts: a 1-minute vertical video about what changed in makable, made by stages that share a folder.

Each stage is a program of its own with its own packages (`shorts.stages`), reading and writing a video's run
folder under the names in `shorts.run`, the contract between them: record (the footage) -> write-script
(Gemini) -> film-intro (Gemini Omni) -> revoice-intro and speak (your voice, on a GPU) -> assemble (ffmpeg).
`shorts.orchestrator` runs what's due, each stage in a process of its own. The other modules are what the
stages use, and know nothing of run folders.
"""


In [ ]:
%%writefile /content/shorts/media.py
"""Small wrappers around ffmpeg and ffprobe."""

import json
import re
import subprocess
from pathlib import Path


def ffmpeg(*args: str | Path) -> None:
    """Runs ffmpeg (overwriting outputs) and raises with its error text if it fails."""
    command = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", *map(str, args)]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed: {result.stderr.strip()}")


def duration(path: Path) -> float:
    """Returns a media file's length in seconds."""
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "json", str(path)],
        capture_output=True,
        text=True,
        check=True,
    )
    return float(json.loads(result.stdout)["format"]["duration"])


def frame_size(path: Path) -> tuple[int, int]:
    """Returns a video's width and height."""
    result = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-select_streams",
            "v:0",
            "-show_entries",
            "stream=width,height",
            "-of",
            "json",
            str(path),
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    stream = json.loads(result.stdout)["streams"][0]
    return stream["width"], stream["height"]


def has_sound(path: Path) -> bool:
    """Whether a media file has an audio stream."""
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=index", "-of", "json", str(path)],
        capture_output=True,
        text=True,
        check=True,
    )
    return bool(json.loads(result.stdout).get("streams"))


def changes(path: Path, fps: int, pixel: int = 8, area: float = 0.15) -> list[float]:
    """Returns when a video's picture changes (seconds into it): each frame in which more than `area` % of
    the pixels got more than `pixel` (of 255) lighter or darker since the frame before."""
    # The changed pixels, averaged over a 45x80 grid (16x16-pixel cells of a 720x1280 frame): ffmpeg does
    # it 8x faster than signalstats. accurate_rnd, or a cell with nothing changed comes out 1, not 0.
    cells = 45 * 80
    result = subprocess.run(
        [
            "ffmpeg",
            "-hide_banner",
            "-loglevel",
            "error",
            "-i",
            str(path),
            "-an",
            "-vf",
            f"format=gray,tblend=all_mode=difference,lutyuv=y='if(gt(val,{pixel}),255,0)',"
            "scale=45:80:flags=area+accurate_rnd",
            "-f",
            "rawvideo",
            "-pix_fmt",
            "gray",
            "-",
        ],
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed: {result.stderr.decode().strip()}")
    grids = result.stdout
    return [
        (i // cells + 1) / fps  # tblend's frame n is how frame n + 1 differs from frame n
        for i in range(0, len(grids), cells)
        if sum(grids[i : i + cells]) / (255 * cells) * 100 > area
    ]


def speech_span(path: Path, noise_db: int = -35, min_silence: float = 0.25) -> tuple[float, float]:
    """Returns when the speech in an audio file starts and ends, skipping silence at either end."""
    result = subprocess.run(
        [
            "ffmpeg",
            "-hide_banner",
            "-i",
            str(path),
            "-af",
            f"silencedetect=noise={noise_db}dB:d={min_silence}",
            "-f",
            "null",
            "-",
        ],
        capture_output=True,
        text=True,
    )
    total = duration(path)
    starts = [float(s) for s in re.findall(r"silence_start: (-?[\d.]+)", result.stderr)]
    ends = [float(s) for s in re.findall(r"silence_end: (-?[\d.]+)", result.stderr)]
    silences = list(zip(starts, ends + [total] * (len(starts) - len(ends)), strict=False))
    begin = silences[0][1] if silences and silences[0][0] <= 0.05 else 0.0
    end = silences[-1][0] if silences and silences[-1][1] >= total - 0.05 else total
    return (begin, end) if end > begin else (0.0, total)


In [ ]:
%%writefile /content/shorts/script.py
"""The script: an LLM turns what changed, or what a recording shows, into what's said and shown."""

from typing import Any

from pydantic import BaseModel, Field
from pydantic.json_schema import SkipJsonSchema

TEXT_MODEL = "gemini-3.8-flash"  # free tier in the Gemini API; on Agent Platform, a fraction of a cent a script
WORDS_PER_SECOND = 2.5  # how fast the narration is spoken, near enough
NARRATION = 50  # seconds of narrated recording after the 10-second intro
# The languages a script can be in, by the code Chatterbox speaks them with, and how they're written.
LANGUAGES = {
    "en": "English",
    "hi": (
        "everyday spoken Hindi, in Devanagari, the way a friendly Indian tech YouTuber talks. The English words "
        "Indians say anyway (website, preview, button, menu) stay English, written in Devanagari (वेबसाइट, प्रीव्यू). "
        "Write {product_name} in Latin letters"
    ),
}
# How the narrator sounds, by name. "hype" is the excited style of Shorts like FactoHolic's: a creator showing a
# friend something they just found, fast, with exclamations and a turn before the best part.
TONES = {
    "friendly": "Write like a person talking: short sentences, plain words",
    "hype": (
        "Write like a creator excitedly showing a friend something they just found: short, punchy sentences, plain "
        "words, an exclamation mark where something good happens, words that pull the viewer in (look, just, wait) "
        'and, before the best part, a turn like "wait, that\'s not even the best part"'
    ),
}

SYSTEM = """You write scripts for 1-minute vertical videos (YouTube Shorts, Instagram Reels) about {product}.

The video has two parts:
1. A 10-second intro: the presenter talks to the camera. `intro_line` is what they say: one hook of 14 to 20 words that makes people want to see the change. `intro_setting` describes the shot in one sentence (place, light, framing), not the person.
2. A narrated screen recording: 4 to 6 `scenes`, 110 to 130 words in all. Each scene is one or two short sentences, said over one recording named in `footage`. Use only the recording names you're given.

{tone}. No hashtags or emojis in anything that's said. End by asking people to try {product_name}.
Also write a YouTube `title` (under 70 characters) and a short `description` (2 or 3 sentences).
Write everything that's said, the `title` and the `description` in {language}."""

STEPS_SYSTEM = """You write scripts for 1-minute vertical videos (YouTube Shorts, Instagram Reels) about {product}.

The video has two parts:
1. A 10-second intro: the presenter talks to the camera. `intro_line` is what they say: one hook of 14 to 20 words that makes people want to see what happens. `intro_setting` describes the shot in one sentence (place, light, framing), not the person.
2. A narrated screen recording: 3 to 8 `scenes`, about {words} words in all. You're given what happens in each recording, step by step, with when (seconds into it). Each scene is said over one stretch of one recording: `footage` names the recording, `start` and `end` (seconds) are where the stretch begins and ends. The narration is spoken at about {rate:g} words a second, so give each scene about {rate:g} words for each second of its stretch (6 seconds: about {six} words), and pick stretches that add up to about {seconds} seconds.

Tell the story in order. The stretches play one after another and never overlap, so leave out what doesn't move the story along: retries, error messages, a name that was already taken, moments where nothing new happens. Say what's on screen while it's on screen.

A program used the site to make the recording, and the steps are in its words ("Click element [471]", "Wait 30 seconds"). Never mention it, its waits or its retries: narrate as if you were showing the viewer. A step that only waits shows what the page did meanwhile (progress, a preview loading), cut down to the moments it changed. Typed text appears on screen as it's typed. You can use facts from the program's final report, such as how long the build took or what the finished page has, but claim nothing the steps and the report don't back up: not "instantly" for something that took 30 seconds.

{tone}. No hashtags or emojis in anything that's said. End by asking people to try {product_name}.
Also write a YouTube `title` (under 70 characters) and a short `description` (2 or 3 sentences).
Write everything that's said, the `title` and the `description` in {language}."""


class Scene(BaseModel):
    text: str = Field(description="What the narrator says over this recording: one or two short sentences.")
    footage: str = Field(description="The name of the recording shown while it's said.")
    # Where in the recording the scene is, in seconds, for a script written from steps. Without them, the
    # recording plays on from where the scene before on it stopped. Not asked for by `write_script`.
    start: SkipJsonSchema[float | None] = None
    end: SkipJsonSchema[float | None] = None


class Script(BaseModel):
    intro_line: str = Field(description="What the presenter says to the camera, 14 to 20 words.")
    intro_setting: str = Field(description="The intro shot in one sentence: place, light, framing.")
    scenes: list[Scene]
    title: str
    description: str
    language: SkipJsonSchema[str] = "en"  # what it's written in (a code in LANGUAGES), so it's spoken in it


class _TimedScene(Scene):
    text: str = Field(description="What the narrator says over this stretch of the recording.")
    start: float = Field(description="Where the stretch begins, in seconds into the recording.")
    end: float = Field(description="Where the stretch ends, in seconds into the recording.")


class _TimedScript(Script):
    scenes: list[_TimedScene]


def _ask(client: Any, model: str, system: str, contents: str, schema: type[BaseModel]) -> Script:
    """Asks the model for a script, as JSON matching `schema`."""
    from google.genai import types

    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system,
            response_mime_type="application/json",
            response_schema=schema,
        ),
    )
    return Script.model_validate_json(response.text)


def written_in(language: str, product_name: str) -> str:
    """How the prompt asks for `language` (a code in `LANGUAGES`)."""
    if language not in LANGUAGES:
        raise ValueError(f"No script language {language!r}: one of {', '.join(LANGUAGES)}")
    return LANGUAGES[language].format(product_name=product_name)


def in_tone(tone: str) -> str:
    """How the prompt asks for `tone` (a name in `TONES`)."""
    if tone not in TONES:
        raise ValueError(f"No tone {tone!r}: one of {', '.join(TONES)}")
    return TONES[tone]


def write_script(
    client: Any,
    changes: str,
    footage: list[str],
    *,
    language: str = "en",
    tone: str = "friendly",
    product: str = "makable, an AI website builder",
    model: str = TEXT_MODEL,
) -> Script:
    """Asks the model for the script, as JSON matching `Script`.

    Args:
        client: A `google.genai.Client`.
        changes: What changed: commit messages, release notes, or a few lines about the feature.
        footage: The names of the recordings the scenes can use.
        language: What it's written (and later spoken) in: a code in `LANGUAGES`, "en" or "hi".
        tone: How it sounds: a name in `TONES`, "friendly" or "hype" (excited, the way Shorts creators talk).
        product: The product, as the narrator would introduce it.
        model: The Gemini text model.
    """
    product_name = product.split(",")[0]
    script = _ask(
        client,
        model,
        SYSTEM.format(
            product=product,
            product_name=product_name,
            language=written_in(language, product_name),
            tone=in_tone(tone),
        ),
        f"What changed:\n{changes}\n\nRecordings you can use: {', '.join(footage)}",
        Script,
    )
    script.language = language
    for index, scene in enumerate(script.scenes):
        if scene.footage not in footage:
            scene.footage = footage[index % len(footage)]
    return script


def recording_length(steps: list[dict[str, Any]]) -> float:
    """How long a recording is, in seconds: until its last step ends."""
    return max((step["end"] for step in steps), default=0.0)


def timeline(name: str, steps: list[dict[str, Any]]) -> str:
    """A recording's steps for the prompt, one line each: when, what the step set out to do, and what it did."""
    lines = [f'Recording "{name}" ({recording_length(steps):.1f} s):']
    for step in steps:
        when = f"{step['start']:.1f}-{step['end']:.1f} s"
        kinds = {kind for action in step["actions"] for kind in action}
        if "done" in kinds:  # its account is the run's final report
            lines.append(f"{when}: the end. The program's final report:\n" + "\n".join(step["did"]))
            continue
        did = "; ".join(" ".join(item.split()) for item in step["did"] if item)
        waits = " (only the moments the page changed are kept)" if kinds == {"wait"} else ""
        lines.append(f"{when}: {step['goal']} Did: {did}{waits}")
    return "\n".join(lines)


def write_script_from_steps(
    client: Any,
    recordings: dict[str, list[dict[str, Any]]],
    *,
    changes: str = "",
    language: str = "en",
    tone: str = "friendly",
    product: str = "makable, an AI website builder",
    model: str = TEXT_MODEL,
) -> Script:
    """Asks the model for a script narrating recordings made by `browse.record_task`, from their steps.

    Each scene is said over a stretch of a recording (`start` to `end`), picked to tell the story and
    sized to the narration, so what's said matches what's on screen. Retries and errors are left out.

    Args:
        client: A `google.genai.Client`.
        recordings: Each recording's name and its steps, as saved in its `.steps.json` (`browse.load_steps`,
            or the JSON itself). Use the steps of the video that's shown: `cut_pauses`' for `-tight.mp4`.
        changes: What changed, if the video is about a change; the steps are enough otherwise.
        language: What it's written (and later spoken) in: a code in `LANGUAGES`, "en" or "hi".
        tone: How it sounds: a name in `TONES`, "friendly" or "hype" (excited, the way Shorts creators talk).
        product: The product, as the narrator would introduce it.
        model: The Gemini text model.
    """
    recordings = {
        name: [step if isinstance(step, dict) else vars(step) for step in steps] for name, steps in recordings.items()
    }
    lengths = {name: recording_length(steps) for name, steps in recordings.items()}
    seconds = round(min(NARRATION, sum(lengths.values())))
    product_name = product.split(",")[0]
    system = STEPS_SYSTEM.format(
        product=product,
        product_name=product_name,
        language=written_in(language, product_name),
        tone=in_tone(tone),
        words=round(seconds * WORDS_PER_SECOND),
        rate=WORDS_PER_SECOND,
        six=round(6 * WORDS_PER_SECOND),
        seconds=seconds,
    )
    contents = "\n\n".join(timeline(name, steps) for name, steps in recordings.items())
    if changes:
        contents = f"What changed:\n{changes}\n\n{contents}"
    script = _ask(client, model, system, contents, _TimedScript)
    script.language = language
    names = list(recordings)
    for index, scene in enumerate(script.scenes):
        if scene.footage not in recordings:
            scene.footage = names[index % len(names)]
        length = lengths[scene.footage]
        scene.start = min(max(scene.start or 0.0, 0.0), length)
        scene.end = min(max(scene.end or length, scene.start), length)
    return script


In [ ]:
%%writefile /content/shorts/intro.py
"""The intro: Gemini Omni Flash films you saying the hook, from a photo of you.

About $0.10 a second at 720p (about $1 a clip). Through Agent Platform (Vertex AI) it's paid from
Google Cloud credits (the Free Trial's, Google AI Pro's) with no prepaid balance; the Gemini API
(AI Studio) only takes those credits while a prepaid balance is above $0. The voice in the clip is
Omni's; `voice.py` turns it into yours.
"""

import base64
import mimetypes
from pathlib import Path
from typing import Any

from .script import Script

VIDEO_MODEL = "gemini-omni-1.1-flash"  # generally available in the Gemini API
VERTEX_VIDEO_MODEL = "gemini-omni-1.1-flash-preview"  # still a preview on Agent Platform


def video_model(client: Any) -> str:
    """Omni's id on the client's backend: Agent Platform (`enterprise=True`) or the Gemini API."""
    return VERTEX_VIDEO_MODEL if client.vertexai else VIDEO_MODEL


def intro_prompt(script: Script) -> str:
    return (
        "Vertical selfie-style video of the person in the photo talking straight to the camera. "
        f"{script.intro_setting} "
        f'They say: "{script.intro_line}" '
        "Natural, friendly delivery. Quiet room, no music. No captions or text on screen."
    )


def film_intro(client: Any, script: Script, face: Path, out: Path, *, model: str | None = None) -> Path:
    """Generates the intro clip (9:16, 720p, with Omni's own voice) and saves it to `out`.

    Args:
        client: A `google.genai.Client` on a project with billing (Omni has no free tier), either
            `genai.Client(enterprise=True, project=..., location="global")` or a Gemini API key's.
        script: The script; its `intro_line` and `intro_setting` make the prompt.
        face: A photo of you, face clearly visible.
        out: Where to save the MP4.
        model: The Omni model; by default the one for the client's backend (`video_model`).

    Raises:
        RuntimeError: Omni returned no video, e.g. when its safety filters refused the photo or prompt.
    """
    interaction = client.interactions.create(
        model=model or video_model(client),
        input=[
            {
                "type": "image",
                "data": base64.b64encode(face.read_bytes()).decode(),
                "mime_type": mimetypes.guess_type(face.name)[0] or "image/jpeg",
            },
            {"type": "text", "text": intro_prompt(script)},
        ],
        response_format={"type": "video", "aspect_ratio": "9:16", "resolution": "720p"},
        timeout=600,
    )
    video = interaction.output_video
    if video is None or not video.data:
        raise RuntimeError(f"Omni returned no video (status {interaction.status}): {interaction.errors}")
    data = video.data
    out.write_bytes(data if isinstance(data, bytes) else base64.b64decode(data))
    return out


In [ ]:
%%writefile /content/shorts/voice.py
"""The voices: every voice in the video becomes yours, from one recording of you.

The intro's speech is converted (same words and timing, so the lips still match) and the narration
is spoken by Chatterbox. Both take the voice from the same file, so they sound like one person.
`OmniVoice` speaks the narration with OmniVoice instead, in an environment of its own.
"""

import functools
import json
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

from .media import ffmpeg

# Words Chatterbox says wrong, by language, spelled the way they're said. Only what it reads changes, not the
# captions. English "makeable" was picked by ear from the voice notebook's tries (section 2). Hindi scripts keep
# makable in Latin letters, so it's read the way a Hindi speaker would write it.
PRONOUNCE = {"en": {"makable": "makeable"}, "hi": {"makable": "मेकेबल"}}
# How the narration is said: livelier than Chatterbox's default (0.5 and 0.5), as its tips suggest for expressive
# speech. More exaggeration is more feeling (and faster); less cfg_weight slows it back down. "lively", picked by
# ear from the voice notebook's tries (section 2).
DELIVERY = {"exaggeration": 0.7, "cfg_weight": 0.3}
# Where `install_omnivoice` puts OmniVoice: chatterbox-tts pins transformers 5.2, and OmniVoice needs 5.3 or newer.
OMNIVOICE_ENV = Path.home() / "omnivoice-env"


@functools.cache
def _tts(device: str, multilingual: bool):
    if multilingual:
        from chatterbox.mtl_tts import ChatterboxMultilingualTTS

        return ChatterboxMultilingualTTS.from_pretrained(device=device)
    from chatterbox.tts import ChatterboxTTS

    return ChatterboxTTS.from_pretrained(device=device)


@functools.cache
def _vc(device: str):
    from chatterbox.vc import ChatterboxVC

    return ChatterboxVC.from_pretrained(device)


@functools.cache
def _omnivoice_here(device: str | None):
    """OmniVoice in this Python, loaded once."""
    from .omnivoice_worker import Speaker

    return Speaker(device)


@functools.cache
def _omnivoice(env: Path, device: str | None) -> tuple[subprocess.Popen, Path]:
    """OmniVoice's worker (omnivoice_worker.py), started once and kept for every scene, and its log."""
    install_omnivoice(env)
    log = Path(tempfile.gettempdir()) / "omnivoice.log"
    command = [env / "bin" / "python", "-I", Path(__file__).with_name("omnivoice_worker.py"), *filter(None, [device])]
    with log.open("w") as stderr:
        process = subprocess.Popen(command, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=stderr, text=True)
    return process, log


def install_omnivoice(env: Path = OMNIVOICE_ENV) -> None:
    """Installs OmniVoice in an environment of its own, with uv: a few minutes the first time, then a moment's check."""
    if not shutil.which("uv"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    python = env / "bin" / "python"
    if not python.exists():
        subprocess.run(["uv", "venv", "--quiet", "--python", "3.12", str(env)], check=True)
    # The CUDA 12.6 build with an NVIDIA GPU (Colab's T4), else the CPU one, which is far smaller.
    build = "cu126" if shutil.which("nvidia-smi") else "cpu"
    install = ["uv", "pip", "install", "--quiet", "--python", str(python)]
    subprocess.run(
        [*install, "torch==2.8.0", "torchaudio==2.8.0", "--index-url", f"https://download.pytorch.org/whl/{build}"],
        check=True,
    )
    # OmniVoice uses transformers' internals: the transformers of its release, not whatever came out since.
    subprocess.run([*install, "omnivoice==0.2.1", "transformers==5.14.1"], check=True)


def best_device() -> str:
    import torch

    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def voice_sample(recording: Path, out: Path, *, start: float = 0.0, length: float = 20.0) -> Path:
    """Makes the voice sample `Voice` takes from a recording of you talking (a video works).

    Its sound in mono, from `start` seconds in, without the silence before you start talking, and at most
    `length` seconds of it: Chatterbox takes the voice from the first 10 seconds or so.
    """
    out.parent.mkdir(parents=True, exist_ok=True)
    ffmpeg(
        "-ss", f"{start:.3f}", "-i", recording, "-vn",
        "-af", "silenceremove=start_periods=1:start_threshold=-40dB",
        "-t", f"{length:.3f}", "-ac", "1", out,
    )  # fmt: skip
    return out


def separate_speech(video: Path, workdir: Path) -> tuple[Path, Path]:
    """Splits a clip's sound into the speech and everything else, with Demucs.

    Returns:
        The speech and the background, as WAV files in `workdir`.
    """
    workdir.mkdir(parents=True, exist_ok=True)
    audio = workdir / f"{video.stem}.wav"
    ffmpeg("-i", video, "-vn", "-ac", "2", "-ar", "44100", audio)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "demucs.separate",
            "--two-stems=vocals",
            "-n",
            "htdemucs",
            "-o",
            str(workdir),
            str(audio),
        ],
        check=True,
    )
    stems = workdir / "htdemucs" / audio.stem
    return stems / "vocals.wav", stems / "no_vocals.wav"


class Voice:
    """Chatterbox's voice cloning (text to speech) and conversion (speech to speech), in your voice.

    It speaks one language: English with Chatterbox's English model, others ("hi" for Hindi) with its
    multilingual one. The models load on first use, which takes a while, and every Voice shares them, so
    trying several deliveries loads them once.
    """

    def __init__(
        self,
        reference: Path,
        language: str = "en",
        device: str | None = None,
        pronounce: dict[str, str] | None = None,
        **tts_options: float,
    ) -> None:
        """
        Args:
            reference: 10 to 20 seconds of you talking, clean, no music.
            language: What it speaks, as Chatterbox's language code: "en", "hi", ...
            device: "cuda", "mps" or "cpu"; the fastest available by default.
            pronounce: Words to respell for Chatterbox, e.g. {"makable": "make-a-bull"}; the language's
                `PRONOUNCE` by default.
            tts_options: Passed to Chatterbox's `generate`, over `DELIVERY`: e.g. `exaggeration=0.9`. For a
                voice sample in another language than `language`, `cfg_weight=0` carries less of its accent over.
        """
        self.reference = Path(reference)
        self.language = language
        self.device = device
        self.pronounce = PRONOUNCE.get(language, {}) if pronounce is None else pronounce
        self.tts_options = {**DELIVERY, **tts_options}

    @property
    def settings(self) -> str:
        """The language and Chatterbox's settings, for naming what's spoken with them."""
        return repr((self.language, sorted(self.tts_options.items())))

    @property
    def tts(self):
        return _tts(self.device or best_device(), self.language != "en")

    @property
    def vc(self):
        return _vc(self.device or best_device())

    def spoken(self, text: str) -> str:
        """`text` as Chatterbox reads it: the words in `pronounce` respelled, keeping a capital letter."""
        for word, said in self.pronounce.items():
            text = re.sub(
                rf"\b{re.escape(word)}\b",
                lambda match, said=said: said[:1].upper() + said[1:] if match[0][0].isupper() else said,
                text,
                flags=re.IGNORECASE,
            )
        return text

    def say(self, text: str, out: Path) -> Path:
        """Speaks `text` in your voice."""
        import torchaudio

        options = self.tts_options if self.language == "en" else {**self.tts_options, "language_id": self.language}
        wav = self.tts.generate(self.spoken(text), audio_prompt_path=str(self.reference), **options)
        torchaudio.save(str(out), wav, self.tts.sr)
        return out

    def convert(self, speech: Path, out: Path) -> Path:
        """Re-voices a recording of speech in your voice, keeping its words and timing."""
        import torchaudio

        wav = self.vc.generate(audio=str(speech), target_voice_path=str(self.reference))
        torchaudio.save(str(out), wav, self.vc.sr)
        return out


class OmniVoice(Voice):
    """A Voice whose narration OmniVoice speaks, from the same voice sample; Chatterbox still converts the intro,
    since OmniVoice only speaks text.

    With `env=None`, OmniVoice runs in this Python (installed here, as in the OmniVoice notebook, which has no
    Chatterbox). Otherwise it runs in an environment of its own, which the first scene to speak installs (a few
    minutes), in a process started then (it loads the model, and downloads it the first time) and kept for the
    rest. Both name what they speak the same way, so scenes spoken in the OmniVoice notebook are used as they are.

    Two models can make one sample sound like two people, so listen to the intro and the first scene together.
    It speaks Hindi too, but learned it from far less speech than English (117 hours to 206,000).
    """

    def __init__(
        self,
        reference: Path,
        language: str = "en",
        reference_text: str | None = None,
        device: str | None = None,
        pronounce: dict[str, str] | None = None,
        env: Path | None = OMNIVOICE_ENV,
        **tts_options: float,
    ) -> None:
        """
        Args:
            reference: 3 to 10 seconds of you talking, clean: OmniVoice clones worse from longer samples
                (`voice_sample(recording, out, length=10)` makes one).
            language: What it speaks, as OmniVoice's language id: "en", "hi", ...
            reference_text: What's said in `reference`; Whisper writes it down when it's not given.
            device: "cuda", "mps" or "cpu", for OmniVoice and Chatterbox; the fastest available by default.
            pronounce: Words to respell, as for Voice; the language's `PRONOUNCE` (picked for Chatterbox) by default.
            env: OmniVoice's own environment (`install_omnivoice`); None: OmniVoice is installed in this Python.
            tts_options: Passed to OmniVoice's `generate`: e.g. `speed=1.1`, or `num_step=16` (faster, a little
                worse). Chatterbox's `DELIVERY` doesn't apply.
        """
        super().__init__(reference, language, device, pronounce)
        self.reference_text = reference_text
        self.env = env if env is None else Path(env)
        self.tts_options = tts_options

    @property
    def settings(self) -> str:
        """OmniVoice's settings, for naming what's spoken with them."""
        return repr(("OmniVoice", self.language, self.reference_text, sorted(self.tts_options.items())))

    def say(self, text: str, out: Path) -> Path:
        """Speaks `text` in your voice, with OmniVoice."""
        request = {
            "text": self.spoken(text),
            "reference": str(self.reference.resolve()),
            "reference_text": self.reference_text,
            "language": self.language,
            "options": self.tts_options,
            "out": str(Path(out).resolve()),
        }
        if self.env is None:
            _omnivoice_here(self.device).say(request)
            return out
        process, log = _omnivoice(self.env, self.device)
        try:
            process.stdin.write(json.dumps(request) + "\n")
            process.stdin.flush()
            answer = process.stdout.readline()
        except BrokenPipeError:
            answer = ""
        if not answer:
            _omnivoice.cache_clear()  # so the next scene starts it again
            raise RuntimeError(f"OmniVoice stopped. The end of its log ({log}):\n{log.read_text()[-3000:]}")
        error = json.loads(answer).get("error")
        if error:
            raise RuntimeError(f"OmniVoice couldn't say {text!r}:\n{error}")
        return out


def mix_intro_audio(speech: Path, background: Path, out: Path, background_volume: float = 0.3) -> Path:
    """Puts the clip's own background sound (quieter) back under the converted speech."""
    ffmpeg(
        "-i", speech, "-i", background,
        "-filter_complex",
        f"[0:a]aresample=48000,aformat=channel_layouts=stereo[s];"
        f"[1:a]aresample=48000,aformat=channel_layouts=stereo,volume={background_volume}[b];"
        "[s][b]amix=inputs=2:duration=longest:normalize=0",
        out,
    )  # fmt: skip
    return out


In [ ]:
%%writefile /content/shorts/omnivoice_worker.py
"""OmniVoice for `voice.OmniVoice`: in the same Python when OmniVoice is installed there (the OmniVoice notebook),
or run by the Python of its own environment (`voice.install_omnivoice`), next to Chatterbox's.

Run as a script, it loads the model once, then reads requests on stdin, one JSON object a line:

    {"text": "...", "reference": "voice.wav", "reference_text": null, "language": "en", "options": {}, "out": "scene.wav"}

writes the speech to "out", and answers on stdout, a line each: {"out": "scene.wav"}, or {"error": "..."}.
Everything else it prints goes to stderr. With an argument ("cuda", "mps" or "cpu"), it runs there.
"""

import hashlib
import json
import sys
import traceback
from pathlib import Path

MODEL = "k2-fsa/OmniVoice"


class Speaker:
    """OmniVoice's model, loaded once, and the voices it took from samples."""

    def __init__(self, device: str | None = None) -> None:
        """
        Args:
            device: "cuda", "mps" or "cpu"; the fastest available by default.
        """
        import omnivoice
        import torch

        if device is None:
            device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
        dtype = torch.float16 if device == "cuda" else torch.float32
        self.model = omnivoice.OmniVoice.from_pretrained(MODEL, device_map=device, dtype=dtype)
        self.prompts = {}  # the voice taken from each sample (and what's said in it), to reuse

    def say(self, request: dict) -> None:
        """Speaks a request's text (keys as in the module's docstring) into its "out" file, as WAV."""
        import soundfile

        reference = Path(request["reference"])
        voice = hashlib.sha256(reference.read_bytes()).hexdigest(), request["reference_text"]
        if voice not in self.prompts:
            self.prompts[voice] = self.model.create_voice_clone_prompt(str(reference), request["reference_text"])
        [audio] = self.model.generate(
            text=request["text"],
            language=request["language"],
            voice_clone_prompt=self.prompts[voice],
            **request["options"],
        )
        soundfile.write(request["out"], audio, self.model.sampling_rate)


def main() -> None:
    answers = sys.stdout
    sys.stdout = sys.stderr  # what OmniVoice and the libraries under it print, so it can't get into the answers

    speaker = Speaker(sys.argv[1] if len(sys.argv) > 1 else None)
    for line in sys.stdin:
        request = json.loads(line)
        try:
            speaker.say(request)
            answer = {"out": request["out"]}
        except Exception:
            answer = {"error": traceback.format_exc()}
        print(json.dumps(answer), file=answers, flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/shorts/footage.py
"""Footage: Playwright records the app in a browser window.

Async because Playwright's sync API can't run inside Colab's (Jupyter's) event loop: in the
notebook `await record_page(...)`, in a script `asyncio.run(record_page(...))`.
"""

import time
from collections.abc import Awaitable, Callable
from pathlib import Path
from typing import Any

from .media import ffmpeg

# Scrolls the page from top to bottom over `ms` milliseconds.
SMOOTH_SCROLL = """(ms) => new Promise(done => {
  const start = performance.now(), from = scrollY;
  const to = document.documentElement.scrollHeight - innerHeight;
  const step = now => {
    const t = Math.min((now - start) / ms, 1);
    scrollTo(0, from + (to - from) * t);
    t < 1 ? requestAnimationFrame(step) : done();
  };
  requestAnimationFrame(step);
})"""


async def record_page(
    url: str,
    out: Path,
    *,
    seconds: float = 15.0,
    demo: Callable[[Any], Awaitable[None]] | None = None,
    size: tuple[int, int] = (720, 1280),
) -> Path:
    """Records a video of `url` in a browser window of `size`, without the blank frames while it loads.

    Playwright records at the window's own size (it never scales up), and `compose` scales the
    recording to the frame: 720x1280 still gets a site's phone layout (narrower than Tailwind's
    768 px `md`), and a landscape size like (1280, 720) is shown whole over a blurred copy.

    Args:
        url: The page to open.
        out: Where to save the MP4.
        seconds: How long to scroll down the page, when there's no `demo`.
        demo: Your own Playwright steps, e.g. `async def demo(page): await page.click("text=Publish")`.
            Without one, the page is scrolled from top to bottom.
        size: The window's width and height, in CSS pixels.
    """
    from playwright.async_api import async_playwright

    out.parent.mkdir(parents=True, exist_ok=True)
    width, height = size
    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch()
        context = await browser.new_context(
            viewport={"width": width, "height": height},
            record_video_dir=str(out.parent / "playwright"),
            record_video_size={"width": width, "height": height},
        )
        started = time.monotonic()
        page = await context.new_page()
        await page.goto(url, wait_until="networkidle")
        loaded = time.monotonic() - started
        if demo is not None:
            await demo(page)
        else:
            await page.wait_for_timeout(800)
            await page.evaluate(SMOOTH_SCROLL, seconds * 1000)
            await page.wait_for_timeout(800)
        await context.close()
        raw = Path(await page.video.path())
        await browser.close()

    ffmpeg("-ss", f"{loaded:.2f}", "-i", raw, "-an", "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", out)
    raw.unlink()
    return out


In [ ]:
%%writefile /content/shorts/captions.py
"""Captions: a few words at a time, spread across each spoken part, drawn with Pillow.

Drawn as images rather than with ffmpeg's subtitles filter, because not every ffmpeg build has
libass (Homebrew's doesn't). The pictures become one video stream through an ffconcat list.
"""

import re
import urllib.request
from dataclasses import dataclass
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont

SIZE = (1080, 1920)  # the frame the captions are drawn for, by default
FONT_URL = "https://github.com/google/fonts/raw/main/ofl/montserrat/Montserrat%5Bwght%5D.ttf"
CENTER_Y = 0.62  # above the title and buttons Shorts and Reels draw over the bottom


@dataclass
class Caption:
    text: str
    start: float
    end: float


def chunk_words(text: str, max_words: int = 3, max_chars: int = 16) -> list[str]:
    """Splits text into short chunks, ending one at every punctuation mark."""
    chunks: list[str] = []
    current: list[str] = []
    for word in text.split():
        if current and (len(current) == max_words or len(" ".join([*current, word])) > max_chars):
            chunks.append(" ".join(current))
            current = []
        current.append(word)
        if re.search(r"[.!?,;:]$", word):
            chunks.append(" ".join(current))
            current = []
    if current:
        chunks.append(" ".join(current))
    return chunks


def timed(text: str, start: float, end: float) -> list[Caption]:
    """Spreads the chunks of `text` over [start, end], each for a time in proportion to its length."""
    chunks = chunk_words(text)
    weights = [len(chunk) + 3 for chunk in chunks]
    captions = []
    at = start
    for chunk, weight in zip(chunks, weights, strict=True):
        length = (end - start) * weight / sum(weights)
        captions.append(Caption(chunk, at, at + length))
        at += length
    return captions


def load_font(cache: Path, size: int = 88) -> ImageFont.FreeTypeFont:
    """Montserrat ExtraBold, downloaded once into `cache`."""
    path = cache / "Montserrat.ttf"
    if not path.exists():
        cache.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(FONT_URL, path)
    font = ImageFont.truetype(str(path), size)
    font.set_variation_by_axes([800])
    return font


def draw(text: str, font: ImageFont.FreeTypeFont, size: tuple[int, int] = SIZE) -> Image.Image:
    """One caption: white with a black outline, wrapped to the frame, on a transparent frame."""
    width, height = size
    image = Image.new("RGBA", size)
    pen = ImageDraw.Draw(image)
    lines: list[str] = []
    for word in text.split():
        if lines and pen.textlength(f"{lines[-1]} {word}", font=font) <= width - 160:
            lines[-1] = f"{lines[-1]} {word}"
        else:
            lines.append(word)
    pen.multiline_text(
        (width / 2, height * CENTER_Y),
        "\n".join(lines),
        font=font,
        anchor="mm",
        align="center",
        spacing=14,
        fill="white",
        stroke_width=9,
        stroke_fill="black",
    )
    return image


def render(captions: list[Caption], workdir: Path, font: ImageFont.FreeTypeFont, size: tuple[int, int] = SIZE) -> Path:
    """Draws each caption and returns an ffconcat list that shows each one for its time.

    Read it with `-f concat -i <list>`; gaps between captions show a blank frame.
    """
    workdir.mkdir(parents=True, exist_ok=True)
    blank = workdir / "blank.png"
    Image.new("RGBA", size).save(blank)
    entries = ["ffconcat version 1.0"]
    at = 0.0
    for index, caption in enumerate(captions):
        if caption.start > at + 0.001:
            entries += [f"file {blank.name}", f"duration {caption.start - at:.3f}"]
        image = workdir / f"{index:03d}.png"
        draw(caption.text, font, size).save(image)
        entries += [f"file {image.name}", f"duration {caption.end - caption.start:.3f}"]
        at = caption.end
    entries.append(f"file {blank.name}")  # the last entry's duration is ignored
    listing = workdir / "captions.ffconcat"
    listing.write_text("\n".join(entries) + "\n")
    return listing


In [ ]:
%%writefile /content/shorts/compose.py
"""Putting the video together with ffmpeg: parts one after another, with captions and music.

A part is footage on screen while something is heard: the intro and its speech, a scene's stretch of a
recording and its narration, or a recording and its own sound. A stretch with an end is sped up or slowed
down to last as long as its part, so what's said is on screen while it's said; without one, the footage
plays at its own speed from `start`, looping if it's shorter. Footage the frame's shape fills it, cropped a
little; any other is shown whole over a blurred, darker copy of itself.
"""

from dataclasses import dataclass
from pathlib import Path

from .media import ffmpeg, frame_size, has_sound

FPS = 30
SIZE = (1080, 1920)  # YouTube Shorts and Instagram Reels
GAP = 0.35  # seconds of pause after each scene's narration
TAIL = 1.0  # extra seconds on the last scene
SAME_SHAPE = 0.05  # how far footage's width:height can be from the frame's (a share of it) and still fill it
FONT_CACHE = Path.home() / ".cache" / "makable-shorts"


@dataclass
class Part:
    """A stretch of the video: `footage` on screen for `length` seconds while `speech` plays from its start."""

    footage: Path
    length: float  # seconds on screen
    speech: Path | None = None  # what's heard (the footage itself, for its own sound); None: silence
    text: str = ""  # what's said, for the captions
    said: tuple[float, float] | None = None  # when in `speech` the words start and end; None: all along
    start: float = 0.0  # where in the footage to start
    end: float | None = None  # where to end, fitting start to end to `length`; None: at the footage's own speed

    @property
    def speed(self) -> float:
        return 1.0 if self.end is None else (self.end - self.start) / self.length


def whole_frames(seconds: float) -> float:
    """Rounds to whole frames, so the parts' lengths add up exactly."""
    return round(seconds * FPS) / FPS


def even(size: tuple[int, int]) -> tuple[int, int]:
    """The nearest frame size at or under `size` that yuv420p can have: even numbers."""
    return size[0] // 2 * 2, size[1] // 2 * 2


def fills(size: tuple[int, int], frame: tuple[int, int]) -> bool:
    """Whether footage of `size` is near enough the frame's shape to fill it, cropped a little."""
    return abs(size[0] / size[1] - frame[0] / frame[1]) <= SAME_SHAPE * frame[0] / frame[1]


def fit(size: tuple[int, int], frame: tuple[int, int]) -> str:
    """The filters that scale footage of `size` to the frame."""
    width, height = frame
    if fills(size, frame):
        return f"scale={width}:{height}:force_original_aspect_ratio=increase,crop={width}:{height}"
    small = f"{width // 4}:{height // 4}"
    blur = max(1, min(12, min(width, height) // 16 - 1))  # boxblur's radius must stay under a quarter of the
    # small copy's size (its colour is at half size): 12 for a 1080x1920 frame
    return (
        f"split[a][b];"
        f"[a]scale={small}:force_original_aspect_ratio=increase,crop={small},boxblur={blur}:2,scale={width}:{height},"
        f"eq=brightness=-0.12[bg];"
        f"[b]scale={width}:{height}:force_original_aspect_ratio=decrease[fg];"
        f"[bg][fg]overlay=(W-w)/2:(H-h)/2"
    )


def render_part(part: Part, out: Path, frame: tuple[int, int] = SIZE) -> Path:
    """Encodes one part's picture (no sound), holding the last frame if the footage runs out."""
    if part.end is None:
        source = ["-stream_loop", "-1", "-ss", f"{part.start:.3f}", "-i", part.footage]
        timing = ""
    else:
        source = ["-ss", f"{part.start:.3f}", "-to", f"{part.end:.3f}", "-i", part.footage]
        timing = f"setpts=(PTS-STARTPTS)/{part.speed:.6f},"
    ffmpeg(
        *source,
        "-filter_complex",
        f"[0:v]{timing}fps={FPS},{fit(frame_size(part.footage), frame)},setsar=1,format=yuv420p,"
        f"tpad=stop_mode=clone:stop_duration={part.length:.3f}[v]",
        "-map", "[v]", "-t", f"{part.length:.3f}", "-an",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", out,
    )  # fmt: skip
    return out


def soundtrack(parts: list[Part], out: Path, music: Path | None = None, music_volume: float = 0.12) -> Path:
    """Lays each part's speech at its start (padded to its length), adds music, and evens out the loudness."""
    inputs: list[str | Path] = []
    chains = []
    for index, part in enumerate(parts):
        if part.speech is not None and has_sound(part.speech):
            inputs += ["-i", part.speech]
        else:
            inputs += ["-f", "lavfi", "-i", "anullsrc=channel_layout=stereo:sample_rate=48000"]
        chains.append(
            f"[{index}:a]aresample=48000,aformat=channel_layouts=stereo,apad,atrim=0:{part.length:.3f}[p{index}]"
        )
    labels = "".join(f"[p{index}]" for index in range(len(parts)))
    graph = ";".join(chains) + f";{labels}concat=n={len(parts)}:v=0:a=1[voice]"
    mixed = "[voice]"
    if music is not None:
        total = sum(part.length for part in parts)
        inputs += ["-stream_loop", "-1", "-i", music]
        graph += (
            f";[{len(parts)}:a]aresample=48000,aformat=channel_layouts=stereo,atrim=0:{total:.3f},"
            f"volume={music_volume},afade=t=out:st={max(total - 2, 0):.3f}:d=2[music]"
            ";[voice][music]amix=inputs=2:duration=first:normalize=0[mixed]"
        )
        mixed = "[mixed]"
    # -14 LUFS is what YouTube and Instagram play at. loudnorm works at 192 kHz, hence the resample.
    graph += f";{mixed}loudnorm=I=-14:TP=-1.5:LRA=11,aresample=48000[out]"
    ffmpeg(*inputs, "-filter_complex", graph, "-map", "[out]", "-c:a", "pcm_s16le", out)
    return out


def compose(
    parts: list[Part],
    workdir: Path,
    out: Path,
    *,
    size: tuple[int, int] = SIZE,
    music: Path | None = None,
    captions: bool = True,
) -> Path:
    """Makes the MP4 from the parts, in order: `size` (even numbers), captions of what's said, and music."""
    workdir.mkdir(parents=True, exist_ok=True)
    pictures = [render_part(part, workdir / f"part-{index}.mp4", size) for index, part in enumerate(parts)]
    listing = workdir / "parts.ffconcat"
    listing.write_text("ffconcat version 1.0\n" + "".join(f"file {picture.name}\n" for picture in pictures))
    audio = soundtrack(parts, workdir / "soundtrack.wav", music)

    inputs: list[str | Path] = ["-f", "concat", "-i", listing, "-i", audio]
    picture = ["-map", "0:v"]
    if captions:
        from .captions import Caption, load_font, render, timed  # Pillow, only for captions

        timings: list[Caption] = []
        at = 0.0
        for part in parts:
            if part.text:
                begin, end = part.said or (0.0, part.length)
                timings += timed(part.text, at + begin, at + min(end, part.length))
            at += part.length
        inputs += ["-f", "concat", "-i", render(timings, workdir / "captions", load_font(FONT_CACHE), size)]
        picture = ["-filter_complex", "[0:v][2:v]overlay=0:0:eof_action=pass,format=yuv420p[v]", "-map", "[v]"]
    ffmpeg(
        *inputs, *picture, "-map", "1:a",
        "-c:v", "libx264", "-preset", "medium", "-crf", "20", "-pix_fmt", "yuv420p", "-r", str(FPS),
        "-c:a", "aac", "-b:a", "192k", "-movflags", "+faststart", "-t", f"{sum(part.length for part in parts):.3f}", out,
    )  # fmt: skip
    return out


In [ ]:
%%writefile /content/shorts/run.py
"""The run folder: the one thing the stages share, and the contract between them.

Each stage is a program of its own (`python -m shorts <stage> RUN`), with its own packages, that reads files
from a video's run folder and writes others there, under these names and in these formats. No stage calls or
imports another, so anything that writes a file under its name can stand in for the stage that makes it:
`python -m shorts import` brings in files made elsewhere (a clip from the Gemini app, scenes spoken by
another tool).

    run.json                         the settings: what the video is made from (yours to edit)
    footage/<name>.mp4               each recording the scenes are said over, and
    footage/<name>.steps.json        what happens in it when (browser-use's steps), if that's known
    script.json                      what's said (`script.Script`): the intro line, and each scene over a recording
    intro/filmed.mp4                 Omni's clip of you saying the intro line, in Omni's voice
    intro/voiced.mp4                 the same in your voice
    intro/speech.wav                 its words alone, for timing the captions (when converted here)
    narration/01.wav, 02.wav, ...    each scene's narration, in your voice
    final.mp4, metadata.json         the video, and what to upload it with
    .records/<product>.json          what made each of a product's files, and from what

A product is the files one stage makes. Its record notes, for each file, a fingerprint of what it was made
from and of the file as written. So a file is out of date when what it would be made from now differs (the
scene's text, the voice sample, a setting), and only that file is made again. A file that changed since it
was written was edited by hand: it's kept as it is.
"""

import contextlib
import hashlib
import json
import os
from collections.abc import Iterator
from pathlib import Path
from typing import Any, Literal

from pydantic import BaseModel

from .script import Script

SETTINGS = "run.json"
SCRIPT = "script.json"
FILMED_INTRO = "intro/filmed.mp4"
VOICED_INTRO = "intro/voiced.mp4"
INTRO_SPEECH = "intro/speech.wav"
FINAL = "final.mp4"
METADATA = "metadata.json"
RECORDS = ".records"

# What each stage makes, in the order they're made.
PRODUCTS = ("footage", "script", "intro", "voiced-intro", "narration", "video")


def footage_file(name: str) -> str:
    return f"footage/{name}.mp4"


def steps_file(name: str) -> str:
    return f"footage/{name}.steps.json"


def narration_file(index: int) -> str:
    """Scene `index`'s narration, counting from 0: narration/01.wav for the first."""
    return f"narration/{index + 1:02d}.wav"


class Settings(BaseModel):
    """What a video is made from (run.json). Paths are relative to the run folder, or absolute."""

    language: str = "en"  # what the script is written and spoken in: a code in script.LANGUAGES
    tone: str = "friendly"  # how the script sounds: a name in script.TONES
    changes: str = ""  # what changed, for a script written without a recording's steps
    footage: dict[str, str] = {}  # each recording, by name: a file to copy in, or a page's URL to record
    script: Literal["gemini", "import"] = "gemini"  # script.json: Gemini writes it, or you bring it in
    intro: Literal["omni", "import"] = "omni"  # intro/filmed.mp4: Gemini Omni films it (about $1), or import
    # intro/voiced.mp4: Omni's speech converted to your voice, brought in, or none (the clip's voice is yours)
    intro_voice: Literal["convert", "import", "as-filmed"] = "convert"
    narrator: Literal["chatterbox", "omnivoice", "import"] = "chatterbox"  # who speaks narration/*.wav
    narrator_options: dict[str, float] = {}  # e.g. Chatterbox's exaggeration, or OmniVoice's speed
    pronounce: dict[str, str] | None = None  # words to respell for the narrator; voice.PRONOUNCE by default
    reference_text: str | None = None  # what's said in the voice sample, for OmniVoice (None: Whisper hears it)
    face: str | None = None  # a photo of you, for Omni
    voice: str | None = None  # a sample of you talking: 10-20 s for Chatterbox, 3-10 s for OmniVoice
    music: str | None = None  # played quietly under the video
    size: tuple[int, int] = (1080, 1920)  # the video's width and height
    captions: bool = True


class Made(BaseModel):
    """A file a stage wrote."""

    by: str  # the stage, as it calls itself: "speak (chatterbox)", "import"
    source: str  # a fingerprint of what it was made from
    sha256: str  # of the file as it was written


class Record(BaseModel):
    """A product's record (.records/<product>.json). Only the stage making the product writes it."""

    files: dict[str, Made] = {}  # by path in the run folder


def fingerprint(*parts: Any) -> str:
    """A short fingerprint of what something is made from (anything JSON can write)."""
    text = json.dumps(parts, sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(text.encode()).hexdigest()[:16]


_hashes: dict[tuple[str, int, int], str] = {}


def file_hash(path: Path) -> str:
    """The SHA-256 of a file's contents, worked out once for each version of the file."""
    stat = path.stat()
    key = (str(path.resolve()), stat.st_size, stat.st_mtime_ns)
    if key not in _hashes:
        digest = hashlib.sha256()
        with path.open("rb") as file:
            while chunk := file.read(1 << 20):
                digest.update(chunk)
        _hashes[key] = digest.hexdigest()
    return _hashes[key]


@contextlib.contextmanager
def writing(path: Path) -> Iterator[Path]:
    """A name to write `path` under, moved into place only if the writing works: a stage that fails halfway
    never leaves a file that looks made. It keeps the extension, for tools that go by it."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.stem}.part{path.suffix}")
    try:
        yield temporary
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


class Run:
    """A video's run folder."""

    def __init__(self, folder: Path | str) -> None:
        self.folder = Path(folder)

    def path(self, name: str) -> Path:
        """A file in the run folder, by its name in the contract."""
        return self.folder / name

    def resolve(self, value: str) -> Path:
        """A path from the settings: relative to the run folder, or absolute."""
        return (self.folder / Path(value).expanduser()).resolve()

    def relative(self, path: Path) -> str:
        """`path` as the settings keep it: relative to the run folder, so the folder works from Drive in Colab
        and from a copy of it elsewhere."""
        return os.path.relpath(Path(path).expanduser().resolve(), self.folder.resolve())

    def settings(self) -> Settings:
        file = self.path(SETTINGS)
        return Settings.model_validate_json(file.read_text()) if file.exists() else Settings()

    def save_settings(self, settings: Settings) -> None:
        self.folder.mkdir(parents=True, exist_ok=True)
        with writing(self.path(SETTINGS)) as out:
            out.write_text(settings.model_dump_json(indent=2) + "\n")

    def script(self) -> Script:
        return Script.model_validate_json(self.path(SCRIPT).read_text())

    def record(self, product: str) -> Record:
        file = self.folder / RECORDS / f"{product}.json"
        return Record.model_validate_json(file.read_text()) if file.exists() else Record()

    def save_record(self, product: str, record: Record) -> None:
        with writing(self.folder / RECORDS / f"{product}.json") as out:
            out.write_text(record.model_dump_json(indent=1) + "\n")


In [ ]:
%%writefile /content/shorts/orchestrator.py
"""The orchestrator: what's made, what's next, and each stage run in a process of its own.

It knows the stages only by what they make and read (`shorts.stages`), and never imports their packages. A
stage it runs is `python -m shorts <stage> RUN`, in the Python set for its packages (SHORTS_<EXTRA>_PYTHON, such
as SHORTS_VOICE_PYTHON=~/chatterbox-env/bin/python), or this one. What it can't run here (a stage whose
packages this machine doesn't have, a paid one, what you bring in) it names, with where it runs, and stops.
"""

import os
import subprocess
import sys
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

from .run import FINAL, PRODUCTS, Run
from .stages import STAGES, Import, NotReady, Stage, execute, producer, states
from .stages.base import MISSING, TO_MAKE

PACKAGE_ROOT = Path(__file__).resolve().parent.parent  # the folder with shorts/ in it, for other Pythons


@dataclass
class Status:
    """Where a product is."""

    product: str
    stage: Stage  # what makes it
    files: dict[str, str] = field(default_factory=dict)  # each planned file's state
    waiting: str = ""  # why it can't be made yet
    needed: bool = True  # whether the video is made from it, with the run's settings

    @property
    def todo(self) -> list[str]:
        return [path for path, state in self.files.items() if state in TO_MAKE]

    @property
    def ready(self) -> bool:
        return not self.needed or (not self.waiting and not self.todo)

    def summary(self) -> str:
        if not self.needed:
            return "not needed, with these settings"
        if self.waiting:
            return self.waiting
        if not self.files:
            return "nothing to make"
        counts = Counter(self.files.values())
        if len(self.files) == 1:
            return next(iter(counts))
        return ", ".join(f"{count} {state}" for state, count in counts.items())


def status(run: Run) -> list[Status]:
    """Every product's status, in the order they're made."""
    settings = run.settings()
    stages = {product: producer(run, product) for product in PRODUCTS}
    needed: set[str] = set()
    wanted = ["video"]
    while wanted:  # what the video is made from, and what that's made from
        product = wanted.pop()
        if product not in needed:
            needed.add(product)
            wanted += stages[product].requires(settings)
    result: list[Status] = []
    ready: dict[str, bool] = {}
    for product in PRODUCTS:
        stage = stages[product]
        if product not in needed:
            result.append(Status(product, stage, needed=False))
            ready[product] = True
            continue
        try:
            entry = Status(product, stage, states(run, product, stage.plan(run)))
        except NotReady as reason:
            entry = Status(product, stage, waiting=str(reason))
        unready = [need for need in stage.requires(settings) if not ready[need]]
        if unready:  # before anything it'd say itself: it may only be missing what they make
            entry.waiting = f"waiting for the {', '.join(unready)}"
        ready[product] = entry.ready
        result.append(entry)
    return result


def advice(run: Run, entry: Status) -> str:
    """The command that makes a product."""
    if isinstance(entry.stage, Import):
        return f"bring it in: python -m shorts import {run.folder} {entry.product} FILE ..."
    command = f"python -m shorts {entry.stage.name} {run.folder}"
    return f"{command} ({entry.stage.cost}; or run --paid)" if entry.stage.cost else command


def python_for(stage: Stage) -> str:
    """The Python that runs a stage: SHORTS_<EXTRA>_PYTHON if it's set, or this one."""
    return os.environ.get(f"SHORTS_{stage.extra.upper()}_PYTHON", "") or sys.executable


def missing_packages(python: str, packages: list[str]) -> list[str]:
    """Which of `packages` a Python can't import."""
    if not packages:
        return []
    check = "import importlib.util, sys; print(*(m for m in sys.argv[1:] if importlib.util.find_spec(m) is None))"
    result = subprocess.run([python, "-c", check, *packages], capture_output=True, text=True)
    return result.stdout.split() if result.returncode == 0 else packages


def run_all(run: Run, *, paid: bool = False) -> bool:
    """Makes what's missing or out of date, in order, each stage in a process of its own, until the video is up
    to date (True) or a stage can't run here (False, saying where it runs)."""
    tried: set[str] = set()
    while True:
        entry = next((entry for entry in status(run) if not entry.ready), None)
        if entry is None:
            print(f"{run.path(FINAL)} is up to date")
            return True
        stage = entry.stage
        if entry.waiting:
            print(f"The {entry.product}: {entry.waiting}")
            return False
        if isinstance(stage, Import) or (stage.cost and not paid):
            print(f"The {entry.product}: {entry.summary()}. Next: {advice(run, entry)}")
            return False
        if stage.name in tried:
            print(f"{stage.name} ran, but the {entry.product} is still {entry.summary()}")
            return False
        python = python_for(stage)
        missing = missing_packages(python, stage.packages(run.settings()))
        if missing:
            print(f"The {entry.product}: {entry.summary()}. Next: {advice(run, entry)}")
            print(f"  It needs {', '.join(missing)}, which {python} doesn't have: {stage.where}")
            return False
        print(f"== {stage.name}: the {entry.product}, {entry.summary()}", flush=True)
        paths = [str(PACKAGE_ROOT), *filter(None, [os.environ.get("PYTHONPATH")])]
        environment = {**os.environ, "PYTHONPATH": os.pathsep.join(paths)}
        if subprocess.run([python, "-m", "shorts", stage.name, str(run.folder)], env=environment).returncode:
            print(f"{stage.name} failed")
            return False
        tried.add(stage.name)


def run_stage(run: Run, name: str, *, force: bool = False) -> list[str]:
    """Runs one stage here, in this process: what `python -m shorts <stage> RUN` does."""
    stage = STAGES[name]
    if producer(run, stage.product).name != stage.name:
        print(f"The {stage.product} is yours, brought in (run.json says so): {name} leaves it as it is")
        return []
    entries = {entry.product: entry for entry in status(run)}
    for need in stage.requires(run.settings()):
        entry = entries[need]
        if (entry.waiting and not entry.files) or MISSING in entry.files.values():
            raise NotReady(f"{name} needs the {need}, which isn't made ({entry.summary()}): {advice(run, entry)}")
        if not entry.ready:
            print(f"Going on with the {need} as it is, though it's {entry.summary()}")
    return execute(run, stage, force=force)


In [ ]:
%%writefile /content/shorts/__main__.py
"""The command line: set a video up, see where it is, and make it, a stage at a time or all that's left.

    python -m shorts new RUN --footage makable=recordings/makable-tight.mp4 --voice assets/voice.wav \\
        --face assets/face.jpg [--music assets/music.mp3] [--language hi] [--tone hype] [--narrator omnivoice]
    python -m shorts status RUN
    python -m shorts run RUN [--paid]          every stage that's due, each in a process of its own
    python -m shorts <stage> RUN [--force]     one stage, here: record, write-script, film-intro,
                                               revoice-intro, speak, assemble
    python -m shorts import RUN PRODUCT FILE ...   files made elsewhere: footage (--name NAME), script,
                                               intro, voiced-intro, narration

The Gemini client is set up from the environment. For Agent Platform (Vertex AI), signed in with
`gcloud auth application-default login` (or, in GitHub Actions, google-github-actions/auth):

    GOOGLE_GENAI_USE_ENTERPRISE=true GOOGLE_CLOUD_PROJECT=<project> GOOGLE_CLOUD_LOCATION=global

For the Gemini API (AI Studio): GEMINI_API_KEY=<key>.
"""

import argparse
import sys
from pathlib import Path

from .orchestrator import advice, run_all, run_stage, status
from .run import SETTINGS, Run, Settings
from .script import LANGUAGES, TONES
from .stages import STAGES, NotReady, bring_in
from .stages.bring_in import PRODUCTS as IMPORTABLE
from .stages.record import is_url


def size(text: str) -> tuple[int, int]:
    """A size such as 1080x1920."""
    try:
        width, height = map(int, text.lower().split("x"))
    except ValueError:
        raise argparse.ArgumentTypeError(f"{text!r} isn't a size like 1080x1920") from None
    return width, height


def option(text: str) -> tuple[str, float]:
    """A narrator option such as exaggeration=0.9."""
    key, _, value = text.partition("=")
    try:
        return key, float(value)
    except ValueError:
        raise argparse.ArgumentTypeError(f"{text!r} isn't an option like exaggeration=0.9") from None


def respelling(text: str) -> tuple[str, str]:
    """A word and how the narrator should read it, such as makable=makeable."""
    word, _, said = text.partition("=")
    if not word or not said:
        raise argparse.ArgumentTypeError(f"{text!r} isn't a respelling like makable=makeable")
    return word, said


def new(run: Run, args: argparse.Namespace) -> None:
    """Writes run.json, or changes the settings given in it."""
    settings = run.settings()
    updates = {key: getattr(args, key) for key in Settings.model_fields if getattr(args, key, None) is not None}
    # (footage, narrator options and respellings come in one at a time, and are added to what's there)
    for key in ("face", "voice", "music"):  # "" for none
        if key in updates:
            updates[key] = run.relative(Path(updates[key])) if updates[key] else None
    if "changes" in updates:
        updates["changes"] = Path(updates["changes"]).read_text()
    for item in args.footage_items or []:
        name, _, source = item.partition("=")
        settings.footage[name] = source if is_url(source) else run.relative(Path(source))
    if args.narrator_option:
        updates["narrator_options"] = {**settings.narrator_options, **dict(args.narrator_option)}
    if args.respellings:
        updates["pronounce"] = {**(settings.pronounce or {}), **dict(args.respellings)}
    run.save_settings(settings.model_copy(update=updates))
    print(run.path(SETTINGS).read_text())


def show_status(run: Run) -> None:
    entries = status(run)
    print(run.folder)
    for entry in entries:
        print(f"  {entry.product:13} {entry.stage.name:14} {entry.summary()}")
    waiting = next((entry for entry in entries if not entry.ready), None)
    print(f"Next: {advice(run, waiting)}" if waiting and not waiting.waiting else "")


def main() -> None:
    parser = argparse.ArgumentParser(prog="python -m shorts", description="Make a 1-minute video, stage by stage.")
    commands = parser.add_subparsers(dest="command", required=True)

    setup = commands.add_parser("new", help="write run.json, or change settings in it")
    setup.add_argument("run", type=Path)
    setup.add_argument("--footage", dest="footage_items", action="append", metavar="NAME=FILE_OR_URL")
    setup.add_argument("--changes", type=Path, help="a text file: what changed, for a script without steps")
    setup.add_argument("--face", help="a photo of you, for Omni")
    setup.add_argument("--voice", help="a sample of you talking")
    setup.add_argument("--music", help='music to play quietly under the video ("" for none)')
    setup.add_argument("--language", choices=sorted(LANGUAGES))
    setup.add_argument("--tone", choices=list(TONES))
    setup.add_argument("--script", choices=["gemini", "import"])
    setup.add_argument("--intro", choices=["omni", "import"])
    setup.add_argument("--intro-voice", choices=["convert", "import", "as-filmed"])
    setup.add_argument("--narrator", choices=["chatterbox", "omnivoice", "import"])
    setup.add_argument("--narrator-option", type=option, action="append", metavar="KEY=VALUE")
    setup.add_argument("--pronounce", dest="respellings", type=respelling, action="append", metavar="WORD=SAID")
    setup.add_argument("--reference-text", help="what's said in the voice sample, for OmniVoice")
    setup.add_argument("--size", type=size, help="the video's width x height (default 1080x1920)")
    setup.add_argument("--captions", action=argparse.BooleanOptionalAction)

    commands.add_parser("status", help="what's made, and what's next").add_argument("run", type=Path)

    everything = commands.add_parser("run", help="every stage that's due, each in a process of its own")
    everything.add_argument("run", type=Path)
    everything.add_argument("--paid", action="store_true", help="film the intro too, if it's due (about $1)")

    imported = commands.add_parser("import", help="bring in files made elsewhere")
    imported.add_argument("run", type=Path)
    imported.add_argument("product", choices=IMPORTABLE)
    imported.add_argument("files", type=Path, nargs="+")
    imported.add_argument("--name", help="the footage's name, as the script calls it")
    imported.add_argument("--as-given", action="store_true", help="narration in the order given, not by number")

    for name, stage in STAGES.items():
        one = commands.add_parser(name, help=f"make the {stage.product}, here")
        one.add_argument("run", type=Path)
        one.add_argument("--force", action="store_true", help="make every file again, edited ones too")

    args = parser.parse_args()
    run = Run(args.run)
    try:
        if args.command == "new":
            new(run, args)
        elif args.command == "status":
            show_status(run)
        elif args.command == "run":
            sys.exit(0 if run_all(run, paid=args.paid) else 1)
        elif args.command == "import":
            bring_in(run, args.product, args.files, name=args.name, as_given=args.as_given)
        else:
            run_stage(run, args.command, force=args.force)
    except (NotReady, ValueError, FileNotFoundError) as problem:
        sys.exit(f"{args.command}: {problem}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/shorts/stages/__init__.py
"""The stages, and which one makes each product for a run (from its run.json).

    record          footage         copies recordings in, or records pages (Playwright)
    write-script    script          Gemini, from the footage's steps or what changed
    film-intro      intro           Gemini Omni films you (about $1)
    revoice-intro   voiced-intro    Demucs and Chatterbox: the intro in your voice (GPU)
    speak           narration       Chatterbox or OmniVoice speaks each scene (GPU)
    assemble        video           ffmpeg and Pillow: final.mp4

What you bring in (`python -m shorts import`) is made by no stage: its product's stage is an `Import`.
"""

from ..run import Run
from .assemble import Assemble
from .base import NotReady, Stage, execute, states
from .bring_in import Import, bring_in
from .film_intro import FilmIntro
from .record import RecordFootage
from .revoice_intro import RevoiceIntro
from .speak import Speak
from .write_script import WriteScript

STAGES: dict[str, Stage] = {
    stage.name: stage for stage in (RecordFootage(), WriteScript(), FilmIntro(), RevoiceIntro(), Speak(), Assemble())
}

__all__ = ["STAGES", "Import", "NotReady", "Stage", "bring_in", "execute", "producer", "states"]


def producer(run: Run, product: str) -> Stage:
    """The stage that makes `product` for this run: an `Import` when run.json says you bring it in."""
    settings = run.settings()
    yours = {"script": settings.script, "intro": settings.intro, "voiced-intro": settings.intro_voice}
    yours["narration"] = settings.narrator
    if yours.get(product) == "import":
        return Import(product)
    return next(stage for stage in STAGES.values() if stage.product == product)


In [ ]:
%%writefile /content/shorts/stages/base.py
"""What every stage is: what it makes, what it reads, and how it plans and makes its files.

A stage plans before it makes. `plan` names its product's files (by their names in the contract, `shorts.run`)
and fingerprints what each would be made from now, without loading anything heavy, so every stage's status
can be worked out on any machine. `make` writes the files it's given. Stage modules import their heavy
packages (torch, Chatterbox, google-genai, Playwright) inside `make` only.
"""

from pathlib import Path

from ..run import SCRIPT, Made, Record, Run, Settings, file_hash
from ..script import Script

# The states of a planned file.
CURRENT = "up to date"
EDITED = "edited by hand"  # changed since its stage wrote it: kept as it is
ELSEWHERE = "made elsewhere"  # there, with no record of what made it: used as it is
MISSING = "missing"
STALE = "out of date"  # what it would be made from now isn't what it was made from
TO_MAKE = {MISSING, STALE}


class NotReady(Exception):
    """A stage can't plan or make its files yet: something they're made from isn't there."""


class Stage:
    """One stage: a program of its own, run as `python -m shorts <name> RUN`."""

    name = ""  # its command
    product = ""  # what it makes: one of run.PRODUCTS
    needs: tuple[str, ...] = ()  # the products it reads (`requires` says which, when the settings decide)
    extra = ""  # its packages: pip install 'makable-shorts[<extra>]'
    where = ""  # where it runs, for a machine without them
    cost = ""  # what each run costs, for a paid stage: `python -m shorts run` never runs one on its own

    def by(self, settings: Settings) -> str:
        """What the stage calls itself in a file's record."""
        return self.name

    def requires(self, settings: Settings) -> tuple[str, ...]:
        """The products it reads, with these settings."""
        return self.needs

    def packages(self, settings: Settings) -> list[str]:
        """The modules it imports, to check whether a Python can run it."""
        return []

    def plan(self, run: Run) -> dict[str, str]:
        """Its product's files (paths in the run folder), each with a fingerprint of what it'd be made from."""
        raise NotImplementedError

    def make(self, run: Run, paths: list[str]) -> None:
        """Writes the files in `paths` (some of what `plan` names)."""
        raise NotImplementedError


def read_script(run: Run) -> Script:
    if not run.path(SCRIPT).exists():
        raise NotReady(f"{SCRIPT} isn't written yet")
    return run.script()


def voice_sample(run: Run, settings: Settings) -> Path:
    if settings.voice is None:
        raise NotReady("run.json has no voice sample: python -m shorts new RUN --voice SAMPLE")
    sample = run.resolve(settings.voice)
    if not sample.exists():
        raise NotReady(f"the voice sample {sample} isn't there")
    return sample


def states(run: Run, product: str, plan: dict[str, str]) -> dict[str, str]:
    """Each planned file's state, from the product's record."""
    record = run.record(product)
    result = {}
    for path, source in plan.items():
        file, made = run.path(path), record.files.get(path)
        if not file.exists():
            result[path] = MISSING
        elif made is None:
            result[path] = ELSEWHERE
        elif file_hash(file) != made.sha256:
            result[path] = EDITED
        elif made.source != source:
            result[path] = STALE
        else:
            result[path] = CURRENT
    return result


def execute(run: Run, stage: Stage, *, force: bool = False) -> list[str]:
    """Makes the stage's files that are missing or out of date (all of them, edited ones too, with `force`),
    and records what they were made from. Returns the files it made."""
    settings = run.settings()
    plan = stage.plan(run)
    todo = [path for path, state in states(run, stage.product, plan).items() if force or state in TO_MAKE]
    if not todo:
        print(f"{stage.product}: {CURRENT}")
        return []

    def written(path: str) -> int | None:
        file = run.path(path)
        return file.stat().st_mtime_ns if file.exists() else None

    before = {path: written(path) for path in plan}
    stage.make(run, todo)
    record = run.record(stage.product)
    files = {path: made for path, made in record.files.items() if path in plan}  # forget what's no longer planned
    for path in plan:  # what it was asked for, and anything else it wrote over
        if path in todo or written(path) != before[path]:
            if not run.path(path).exists():
                raise RuntimeError(f"{stage.name} didn't write {path}")
            files[path] = Made(by=stage.by(settings), source=plan[path], sha256=file_hash(run.path(path)))
    run.save_record(stage.product, Record(files=files))
    return todo


In [ ]:
%%writefile /content/shorts/stages/record.py
"""record: the footage. A file named in run.json is copied in (with its steps, if browser-use made it), and a
page's URL is recorded by Playwright, scrolling down it in a 720x1280 window.

browser-use's recordings come from `record_with_browser_use.ipynb` (its packages clash with the voices'): name
the file it made in run.json, and this copies it with its .steps.json.
"""

import shutil

from ..run import Run, Settings, file_hash, fingerprint, footage_file, steps_file, writing
from .base import NotReady, Stage


def is_url(source: str) -> bool:
    return "://" in source


class RecordFootage(Stage):
    name = "record"
    product = "footage"
    extra = "record"
    where = "pip install 'makable-shorts[record]' && playwright install chromium (or makable_shorts.ipynb in Colab)"

    def packages(self, settings: Settings) -> list[str]:
        return ["playwright"] if any(is_url(source) for source in settings.footage.values()) else []

    def plan(self, run: Run) -> dict[str, str]:
        settings = run.settings()
        if not settings.footage:
            raise NotReady("run.json names no footage: python -m shorts new RUN --footage NAME=FILE_OR_URL")
        plan = {}
        for name, source in settings.footage.items():
            if is_url(source):
                plan[footage_file(name)] = fingerprint("record", source)
                continue
            video = run.resolve(source)
            if not video.exists():
                raise NotReady(f"the footage {name!r} is {video}, which isn't there")
            plan[footage_file(name)] = fingerprint("copy", file_hash(video))
            steps = video.with_suffix(".steps.json")  # where browse.record_task saves them
            if steps.exists():
                plan[steps_file(name)] = fingerprint("copy", file_hash(steps))
        return plan

    def make(self, run: Run, paths: list[str]) -> None:
        for name, source in run.settings().footage.items():
            if is_url(source):
                if footage_file(name) in paths:
                    import asyncio

                    from ..footage import record_page

                    print(f"Recording {source}")
                    with writing(run.path(footage_file(name))) as out:
                        asyncio.run(record_page(source, out))
                continue
            video = run.resolve(source)
            for path, original in ((footage_file(name), video), (steps_file(name), video.with_suffix(".steps.json"))):
                if path in paths:
                    print(f"Copying {original} to {path}")
                    with writing(run.path(path)) as out:
                        shutil.copyfile(original, out)


In [ ]:
%%writefile /content/shorts/stages/write_script.py
"""write-script: Gemini writes script.json, from the footage's steps when it has them (each scene then says
what's on screen over its stretch of a recording), or else from what changed (`changes` in run.json).

It's written in run.json's language and tone. Edit script.json to change what's said: an edited script is
kept, even when the settings change (delete it to write a new one).
"""

import json
from pathlib import Path

from ..run import SCRIPT, Run, Settings, file_hash, fingerprint, steps_file, writing
from ..script import TEXT_MODEL
from .base import NotReady, Stage


class WriteScript(Stage):
    name = "write-script"
    product = "script"
    needs = ("footage",)
    extra = "gemini"
    where = "pip install 'makable-shorts[gemini]', with GEMINI_API_KEY or Agent Platform's settings (see the README)"

    def packages(self, settings: Settings) -> list[str]:
        return ["google.genai"]

    def steps(self, run: Run, settings: Settings) -> dict[str, Path]:
        """The footage's steps files, by its name, for the footage that has them."""
        files = {name: run.path(steps_file(name)) for name in settings.footage}
        return {name: path for name, path in files.items() if path.exists()}

    def plan(self, run: Run) -> dict[str, str]:
        settings = run.settings()
        steps = {name: file_hash(path) for name, path in self.steps(run, settings).items()}
        if not steps and not settings.changes:
            raise NotReady("no footage has steps, and run.json has no changes: what's the video about?")
        made_from = (TEXT_MODEL, settings.language, settings.tone, settings.changes, sorted(settings.footage), steps)
        return {SCRIPT: fingerprint("gemini", *made_from)}

    def make(self, run: Run, paths: list[str]) -> None:
        from google import genai

        from .. import script as scripts

        settings = run.settings()
        client = genai.Client()  # Agent Platform or the Gemini API, from the environment (see the README)
        options = {"language": settings.language, "tone": settings.tone}
        steps = {name: json.loads(path.read_text()) for name, path in self.steps(run, settings).items()}
        if steps:
            script = scripts.write_script_from_steps(client, steps, changes=settings.changes, **options)
        else:
            script = scripts.write_script(client, settings.changes, list(settings.footage), **options)
        with writing(run.path(SCRIPT)) as out:
            out.write_text(script.model_dump_json(indent=2) + "\n")
        for index, scene in enumerate(script.scenes, 1):
            print(f"{index}. {scene.text}")


In [ ]:
%%writefile /content/shorts/stages/film_intro.py
"""film-intro: Gemini Omni Flash films you saying the script's intro line, from a photo of you (about $1 a clip).

Because it costs money, `python -m shorts run` never films it on its own: it says when the intro is missing or
out of date (the intro line, its setting or the photo changed), and `python -m shorts film-intro RUN` films it.
A clip filmed elsewhere (the Gemini app) comes in with `python -m shorts import RUN intro CLIP`.
"""

from ..intro import film_intro, intro_prompt
from ..run import FILMED_INTRO, Run, Settings, file_hash, fingerprint, writing
from .base import NotReady, Stage, read_script


class FilmIntro(Stage):
    name = "film-intro"
    product = "intro"
    needs = ("script",)
    extra = "gemini"
    where = "pip install 'makable-shorts[gemini]', on a Google Cloud project with billing (see the README)"
    cost = "about $1"

    def packages(self, settings: Settings) -> list[str]:
        return ["google.genai"]

    def plan(self, run: Run) -> dict[str, str]:
        settings = run.settings()
        if settings.face is None:
            raise NotReady("run.json has no photo of you: python -m shorts new RUN --face PHOTO")
        face = run.resolve(settings.face)
        if not face.exists():
            raise NotReady(f"the photo {face} isn't there")
        return {FILMED_INTRO: fingerprint("omni", intro_prompt(read_script(run)), file_hash(face))}

    def make(self, run: Run, paths: list[str]) -> None:
        from google import genai

        print("Filming the intro (about $1)")
        with writing(run.path(FILMED_INTRO)) as out:
            film_intro(genai.Client(), run.script(), run.resolve(run.settings().face), out)


In [ ]:
%%writefile /content/shorts/stages/revoice_intro.py
"""revoice-intro: the intro in your voice. Demucs separates the speech in Omni's clip from its background,
Chatterbox converts the speech to your voice (same words and timing, so the lips still match), and the
background goes back under it, quieter.

With `intro_voice` "as-filmed" in run.json (the voice sample came from the intro, so it's yours already),
there's nothing to do, and the video uses the clip as Omni filmed it.
"""

import shutil
import tempfile
from pathlib import Path

from ..media import ffmpeg
from ..run import FILMED_INTRO, INTRO_SPEECH, VOICED_INTRO, Run, Settings, file_hash, fingerprint, writing
from .base import NotReady, Stage, voice_sample


class RevoiceIntro(Stage):
    name = "revoice-intro"
    product = "voiced-intro"
    needs = ("intro",)
    extra = "voice"
    where = "a GPU: makable_shorts.ipynb or voice_from_script.ipynb in Colab (or pip install 'makable-shorts[voice]')"

    def requires(self, settings: Settings) -> tuple[str, ...]:
        return self.needs if settings.intro_voice == "convert" else ()

    def packages(self, settings: Settings) -> list[str]:
        return [] if settings.intro_voice == "as-filmed" else ["chatterbox", "demucs", "torchaudio"]

    def plan(self, run: Run) -> dict[str, str]:
        settings = run.settings()
        if settings.intro_voice == "as-filmed":
            return {}
        filmed = run.path(FILMED_INTRO)
        if not filmed.exists():
            raise NotReady(f"{FILMED_INTRO} isn't there yet")
        source = fingerprint("convert", file_hash(filmed), file_hash(voice_sample(run, settings)))
        return {VOICED_INTRO: source, INTRO_SPEECH: source}

    def make(self, run: Run, paths: list[str]) -> None:
        from ..voice import Voice, mix_intro_audio, separate_speech

        settings = run.settings()
        filmed = run.path(FILMED_INTRO)
        voice = Voice(voice_sample(run, settings), settings.language)
        with tempfile.TemporaryDirectory() as work:
            print("Separating the intro's speech from its background (Demucs)")
            speech, background = separate_speech(filmed, Path(work))
            print("Converting the speech to your voice (Chatterbox)")
            converted = voice.convert(speech, Path(work) / "speech.wav")
            audio = mix_intro_audio(converted, background, Path(work) / "audio.wav")
            with writing(run.path(INTRO_SPEECH)) as out:
                shutil.copyfile(converted, out)
            with writing(run.path(VOICED_INTRO)) as out:
                ffmpeg(
                    "-i", filmed, "-i", audio, "-map", "0:v", "-map", "1:a",
                    "-c:v", "copy", "-c:a", "aac", "-shortest", out,
                )  # fmt: skip


In [ ]:
%%writefile /content/shorts/stages/speak.py
"""speak: each scene's narration in your voice, narration/01.wav and on, by Chatterbox or OmniVoice (`narrator`
in run.json, with its settings in `narrator_options`).

Only the scenes whose spoken text, voice sample or settings changed are spoken again. Where OmniVoice runs
comes from the environment, since it's the machine's business, not the video's: OMNIVOICE_ENV is the folder
of its own environment (~/omnivoice-env by default, installed the first time), or "here" for this Python
(OmniVoice installed in it, as in omnivoice_from_script.ipynb).
"""

import os
from pathlib import Path

from ..run import Run, Settings, file_hash, fingerprint, narration_file, writing
from ..voice import OMNIVOICE_ENV, OmniVoice, Voice
from .base import Stage, read_script, voice_sample


def omnivoice_env() -> Path | None:
    value = os.environ.get("OMNIVOICE_ENV", "")
    return None if value == "here" else Path(value).expanduser() if value else OMNIVOICE_ENV


class Speak(Stage):
    name = "speak"
    product = "narration"
    needs = ("script",)
    extra = "voice"
    where = "a GPU: makable_shorts.ipynb, voice_from_script.ipynb or omnivoice_from_script.ipynb in Colab"

    def by(self, settings: Settings) -> str:
        return f"speak ({settings.narrator})"

    def packages(self, settings: Settings) -> list[str]:
        if settings.narrator == "omnivoice":
            return ["omnivoice"] if omnivoice_env() is None else []  # its own environment installs itself
        return ["chatterbox", "torchaudio"]

    def voice(self, run: Run) -> Voice:
        """The narrator, set up (its model loads when it first speaks)."""
        settings = run.settings()
        sample, language = voice_sample(run, settings), read_script(run).language
        options = {"pronounce": settings.pronounce, **settings.narrator_options}
        if settings.narrator == "omnivoice":
            return OmniVoice(sample, language, settings.reference_text, env=omnivoice_env(), **options)
        return Voice(sample, language, **options)

    def plan(self, run: Run) -> dict[str, str]:
        settings = run.settings()
        voice = self.voice(run)
        yours = file_hash(voice.reference)
        return {
            narration_file(index): fingerprint(settings.narrator, voice.settings, yours, voice.spoken(scene.text))
            for index, scene in enumerate(read_script(run).scenes)
        }

    def make(self, run: Run, paths: list[str]) -> None:
        voice = self.voice(run)
        for index, scene in enumerate(run.script().scenes):
            if narration_file(index) in paths:
                print(f"Scene {index + 1}: {scene.text}")
                with writing(run.path(narration_file(index))) as out:
                    voice.say(scene.text, out)


In [ ]:
%%writefile /content/shorts/stages/assemble.py
"""assemble: final.mp4 and metadata.json. The intro in your voice, then each scene's stretch of its recording
while its narration plays (sped up or slowed down to fit it), with captions and music, at -14 LUFS.

A scene without a stretch (a script written from what changed, not from steps) plays its recording on from
where the scene before on it stopped. Needs only ffmpeg, and Pillow for the captions.
"""

import json
import tempfile
from pathlib import Path

from ..compose import GAP, TAIL, Part, compose, even, whole_frames
from ..media import duration, speech_span
from ..run import (
    FILMED_INTRO,
    FINAL,
    INTRO_SPEECH,
    METADATA,
    VOICED_INTRO,
    Run,
    Settings,
    file_hash,
    fingerprint,
    footage_file,
    narration_file,
    writing,
)
from ..script import Script
from .base import NotReady, Stage, read_script


class Assemble(Stage):
    name = "assemble"
    product = "video"
    needs = ("footage", "script", "voiced-intro", "narration")  # or the intro as filmed, in place of voiced-intro
    extra = "assemble"
    where = "pip install 'makable-shorts[assemble]' (Pillow), with ffmpeg"

    def requires(self, settings: Settings) -> tuple[str, ...]:
        if settings.intro_voice == "as-filmed":
            return tuple("intro" if need == "voiced-intro" else need for need in self.needs)
        return self.needs

    def packages(self, settings: Settings) -> list[str]:
        return ["PIL"] if settings.captions else []

    def inputs(self, run: Run, settings: Settings, script: Script) -> dict[str, Path]:
        """The files the video is made from, by what they are."""
        files = {"the intro": run.path(FILMED_INTRO if settings.intro_voice == "as-filmed" else VOICED_INTRO)}
        if settings.intro_voice == "convert":
            files["the intro's speech"] = run.path(INTRO_SPEECH)
        for name in sorted({scene.footage for scene in script.scenes}):
            files[f"the footage {name!r}"] = run.path(footage_file(name))
        for index in range(len(script.scenes)):
            files[f"scene {index + 1}'s narration"] = run.path(narration_file(index))
        if settings.music:
            files["the music"] = run.resolve(settings.music)
        return files

    def plan(self, run: Run) -> dict[str, str]:
        settings, script = run.settings(), read_script(run)
        files = self.inputs(run, settings, script)
        missing = [what for what, path in files.items() if not path.exists()]
        if missing:
            raise NotReady(f"missing {', '.join(missing)}")
        hashes = {what: file_hash(path) for what, path in files.items()}
        source = fingerprint(script.model_dump(), hashes, settings.size, settings.captions, GAP, TAIL)
        return {FINAL: source, METADATA: source}

    def parts(self, run: Run, settings: Settings, script: Script) -> list[Part]:
        files = self.inputs(run, settings, script)
        intro = files["the intro"]
        words = files.get("the intro's speech", intro)
        parts = [Part(intro, whole_frames(duration(intro)), intro, script.intro_line, speech_span(words))]
        played: dict[str, float] = {}  # how far into each recording the scenes without a stretch got
        for index, scene in enumerate(script.scenes):
            footage, speech = run.path(footage_file(scene.footage)), run.path(narration_file(index))
            length = whole_frames(duration(speech) + GAP + (TAIL if index == len(script.scenes) - 1 else 0))
            part = Part(footage, length, speech, scene.text, speech_span(speech))
            if scene.start is not None and scene.end is not None:
                part.start, part.end = scene.start, min(scene.end, duration(footage))
                if part.end - part.start < 0.1:
                    raise NotReady(f"scene {index + 1}'s stretch ({scene.start}-{scene.end} s) isn't in {footage.name}")
            else:
                part.start = played.get(scene.footage, 0.0) % duration(footage)
                played[scene.footage] = part.start + length
            stretch = f"{part.start:5.1f}-{part.end:5.1f} s" if part.end is not None else f"from {part.start:.1f} s"
            print(f"Scene {index + 1}: {scene.footage} {stretch} at {part.speed:.2f}x, {length:.2f} s: {scene.text}")
            parts.append(part)
        return parts

    def make(self, run: Run, paths: list[str]) -> None:
        settings, script = run.settings(), run.script()
        parts = self.parts(run, settings, script)
        music = run.resolve(settings.music) if settings.music else None
        with tempfile.TemporaryDirectory() as work, writing(run.path(FINAL)) as out:
            compose(parts, Path(work), out, size=even(settings.size), music=music, captions=settings.captions)
            seconds = duration(out)
        metadata = {
            "title": script.title,
            "description": script.description,
            # YouTube asks for this when AI makes a real person appear to say something.
            "containsSyntheticMedia": True,
            "seconds": round(seconds, 2),
        }
        with writing(run.path(METADATA)) as out:
            out.write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + "\n")
        print(f"Made {run.path(FINAL)} ({seconds:.1f} s)")


In [ ]:
%%writefile /content/shorts/stages/bring_in.py
"""import: files made elsewhere, brought in under their names in the contract, in place of a stage's.

    python -m shorts import RUN footage --name makable recordings/makable-tight.mp4   (and its .steps.json)
    python -m shorts import RUN script script.json
    python -m shorts import RUN intro gemini_generated_video.mp4          Omni's clip, in Omni's voice
    python -m shorts import RUN voiced-intro intro-in-your-voice.mp4      the intro in your voice already
    python -m shorts import RUN narration download*.wav                   one audio file per scene

Bringing a product in sets run.json so that it's yours from then on (`narrator: "import"`, say), and the
pipeline asks for it again instead of making it. Imported narration is out of date when its scene's text
changes, as speech for it would be.

Narration files are put in order by the numbers in their names (download.wav, download (1).wav, ...,
download (10).wav, as browsers name repeated downloads), or kept as given with `as_given`. A file that's the
same as one before it, downloaded twice, is left out.
"""

import hashlib
import re
import shutil
from pathlib import Path

from ..media import ffmpeg
from ..run import FILMED_INTRO, SCRIPT, VOICED_INTRO, Run, Settings, fingerprint, narration_file, writing
from ..script import Script
from .base import NotReady, Stage, execute, read_script
from .record import RecordFootage

SINGLE = {"script": SCRIPT, "intro": FILMED_INTRO, "voiced-intro": VOICED_INTRO}
PRODUCTS = ("footage", *SINGLE, "narration")


def in_order(files: list[Path]) -> list[Path]:
    """By the numbers in their names: download.wav, download (1).wav, ..., download (10).wav."""

    def key(path: Path) -> list[int | str]:
        return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", path.stem)]

    return sorted(files, key=key)


def without_copies(files: list[Path]) -> list[Path]:
    """`files` without any that's the same as one before it."""
    kept: list[Path] = []
    seen: dict[bytes, Path] = {}
    for path in files:
        first = seen.setdefault(hashlib.sha256(path.read_bytes()).digest(), path)
        if first is path:
            kept.append(path)
        else:
            print(f"Leaving out {path.name}: it's the same as {first.name}")
    return kept


class Import(Stage):
    """A product you bring in: `plan` says what the files should be, and `make` copies `files` in."""

    name = "import"

    def __init__(self, product: str, files: list[Path] | None = None) -> None:
        self.product = product
        self.files = files

    def requires(self, settings: Settings) -> tuple[str, ...]:
        return ("script",) if self.product == "narration" else ()  # one file for each of its scenes

    def plan(self, run: Run) -> dict[str, str]:
        if self.product == "narration":
            return {
                narration_file(index): fingerprint("import", scene.text)
                for index, scene in enumerate(read_script(run).scenes)
            }
        return {SINGLE[self.product]: fingerprint("import")}

    def make(self, run: Run, paths: list[str]) -> None:
        if self.files is None:
            raise NotReady(f"bring it in: python -m shorts import {run.folder} {self.product} FILE ...")
        sources = dict(zip(self.plan(run), self.files, strict=True))
        for path in paths:
            print(f"{sources[path]} -> {path}")
            with writing(run.path(path)) as out:
                if self.product == "narration":
                    ffmpeg("-i", sources[path], "-vn", "-c:a", "pcm_s16le", out)  # any audio file, as WAV
                else:
                    shutil.copyfile(sources[path], out)


def bring_in(run: Run, product: str, files: list[Path], *, name: str | None = None, as_given: bool = False) -> None:
    """Brings `files` in as `product` (see the module's docstring), and makes run.json say they're yours."""
    if product not in PRODUCTS:
        raise ValueError(f"No product {product!r}: one of {', '.join(PRODUCTS)}")
    files = [Path(file).expanduser() for file in files]
    for file in files:
        if not file.is_file():
            raise FileNotFoundError(file)
    settings = run.settings()
    if product == "footage":
        if name is None or len(files) != 1:
            raise ValueError("Bring footage in one recording at a time, with its name: --name NAME FILE")
        settings.footage[name] = run.relative(files[0])
        run.save_settings(settings)
        execute(run, RecordFootage())
        return
    if product == "narration":
        files = without_copies(files if as_given else in_order(files))
        scenes = read_script(run).scenes
        if len(files) != len(scenes):
            raise ValueError(f"The script has {len(scenes)} scenes, but there are {len(files)} audio files")
        for index, (scene, file) in enumerate(zip(scenes, files, strict=True), 1):
            print(f"Scene {index}: {file.name}: {scene.text}")
    elif len(files) != 1:
        raise ValueError(f"Bring in one file as the {product}")
    if product == "script":
        Script.model_validate_json(files[0].read_text())  # before it replaces the one there
    setting = {"script": "script", "intro": "intro", "voiced-intro": "intro_voice", "narration": "narrator"}[product]
    setattr(settings, setting, "import")
    run.save_settings(settings)
    execute(run, Import(product, files), force=True)


In [ ]:
%load_ext autoreload
%autoreload 2
sys.path.insert(0, "/content")

from IPython.display import Audio, Video, display

from shorts.run import FILMED_INTRO, VOICED_INTRO, Run, file_hash, fingerprint, narration_file
from shorts.script import WORDS_PER_SECOND
from shorts.voice import Voice

run = Run(RUN)
TRIES = run.folder / "tries"  # takes to choose from (section 2), apart from the video's own files


def take(voice, text):
    """`voice` saying `text`: said once, and kept in TRIES."""
    out = TRIES / f"{fingerprint(voice.settings, voice.spoken(text), file_hash(voice.reference))}.wav"
    if not out.exists():
        TRIES.mkdir(parents=True, exist_ok=True)
        voice.say(text, out)
    return out


import codecs
import subprocess


def shorts(*args):
    """Runs `python -m shorts ARGS`: a stage, or `new` (settings), `status`, `import`, `run` (all that's due)."""
    command = [sys.executable, "-m", "shorts", *map(str, args)]
    process = subprocess.Popen(command, cwd="/content", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    text = codecs.getincrementaldecoder("utf-8")(errors="replace")
    while chunk := process.stdout.read1(4096):
        print(text.decode(chunk), end="", flush=True)
    if process.wait():
        raise RuntimeError(f"python -m shorts {args[0]} stopped: see above")


## The voice

From `RECORDING`'s sound: up to 20 s, from where the talking starts, saved in `assets/` as `voice-<recording>.wav`. Listen to it: it should be one person talking and nothing else, because Chatterbox copies the room and any background sound along with the voice. With `RECORDING = None`, the voice is `assets/voice.wav`, the Shorts notebook's.

In [ ]:
# The voice: up to 20 s of a recording's sound, from where the talking starts, made once and kept on Drive.
# RECORDING: a video or audio file (a video of you, or the intro Gemini Omni filmed), or a pattern such as
# HOME / "runs" / "*.mp4" (the newest match). None: assets/voice.wav, made the first time from the video in
# assets/ or makable-shorts/, or from a recording you upload. To make a sample again (say, from further in), delete it.
import glob

from google.colab import files

from shorts.media import duration
from shorts.voice import voice_sample

RECORDING = HOME / "crumb-and-co" / "runs" / "intro-*.mp4"
START = 0.0  # seconds into the recording to start from, if it doesn't start with the talking
VIDEOS = {".mp4", ".mov", ".m4v", ".webm", ".mkv", ".avi", ".3gp"}

VOICE_SAMPLE = ASSETS / "voice.wav"
if RECORDING is not None:
    matches = sorted(glob.glob(str(RECORDING)), key=os.path.getmtime)
    if not matches:
        raise FileNotFoundError(f"Nothing on Drive matches {RECORDING}")
    RECORDING = Path(matches[-1])
    VOICE_SAMPLE = ASSETS / f"voice-{RECORDING.stem}.wav"  # one sample per recording
if not VOICE_SAMPLE.exists():
    if RECORDING is None:
        videos = [path for folder in (ASSETS, HOME) if folder.is_dir() for path in sorted(folder.iterdir())]
        videos = [path for path in videos if path.suffix.lower() in VIDEOS]
        if len(videos) > 1:
            raise ValueError("Which video is you? Set RECORDING to one of:\n" + "\n".join(map(str, videos)))
        if videos:
            RECORDING = videos[0]
        else:
            print(f"No video in {ASSETS} or {HOME}. Choose a recording of you talking (any audio or video file):")
            uploads = files.upload()  # saved in /content
            if not uploads:
                raise FileNotFoundError(f"No recording: put a video of you in {ASSETS}, or set RECORDING")
            RECORDING = next(iter(uploads))
    voice_sample(Path(RECORDING), VOICE_SAMPLE, start=START)
    print(f"Made {VOICE_SAMPLE.name} from {RECORDING}")
print(f"The voice: {VOICE_SAMPLE} ({duration(VOICE_SAMPLE):.1f} s). Listen: it should be one person talking, nothing else.")
display(Audio(str(VOICE_SAMPLE)))
# The models load the first time they're needed. Chatterbox speaks LANGUAGE, reads the words in its
# shorts.voice.PRONOUNCE respelled (English "makable" as "makeable"), as lively as shorts.voice.DELIVERY says
# (exaggeration 0.7, cfg_weight 0.3). To change them: Voice(..., pronounce={"makable": "…"}, exaggeration=0.9)
voice = Voice(VOICE_SAMPLE, language=LANGUAGE)


## 1. The script

Paste a script's JSON (a `script.json` the Shorts notebook wrote, or one written from a recording's steps), or set `SCRIPT` to a script.json on Drive. These two were written from `makable-tight.steps.json`, in English and in Hindi; `LANGUAGE` picks one.

If the run's folder has no `script.json` yet, the script is brought in there, so the Shorts notebook goes on with it. The voice sample goes in the run's `run.json`, for the narration.

In [ ]:
SCRIPTS = {
    "en": """
{
  "intro_line": "Watch me build and customize a complete bakery website with an AI builder in under one minute.",
  "intro_setting": "A bright modern kitchen with morning sunlight streaming through large windows, framed in a medium close-up.",
  "scenes": [
    {"text": "We open Makable, click New Site, and start building a website for a local bakery.", "footage": "makable", "start": 0.0, "end": 5.8},
    {"text": "We type our prompt for a hero banner, six priced pastries, opening hours, and contact info.", "footage": "makable", "start": 5.8, "end": 11.6},
    {"text": "We name the bakery Crumb and Co Shop, and Makable gets straight to work generating the layout step by step.", "footage": "makable", "start": 18.8, "end": 27.5},
    {"text": "The live preview loads up with six priced pastries from sourdough to croissants, opening hours, and bakery contact info.", "footage": "makable", "start": 27.5, "end": 36.8},
    {"text": "To refine the look, we ask the chat to switch the hero background to warm cream and add a pickup button.", "footage": "makable", "start": 54.0, "end": 62.9},
    {"text": "The preview updates with our warm cream header and new order button. Head over and try Makable today.", "footage": "makable", "start": 62.9, "end": 69.5}
  ],
  "title": "Building a Bakery Website with AI in Seconds",
  "description": "See how Makable builds a complete bakery website from a single prompt with menu pricing, hours, and contact info. Watch us refine the design with a custom color palette and order button. Try Makable today to create your own website."
}
""",
    "hi": """
{
  "intro_line": "क्या एआई सच में सिर्फ एक प्रॉम्प्ट से पूरी बेकरी वेबसाइट तैयार कर सकता है? चलिए आज लाइव देखते हैं!",
  "intro_setting": "Modern studio desk with warm ambient lighting and soft neon accents, framed in a tight vertical shot.",
  "scenes": [
    {"text": "सबसे पहले मेन्यू से न्यू साइट चुनते हैं और प्रॉम्प्ट में अपनी बेकरी का पूरा मेन्यू, टाइमिंग और कॉन्टैक्ट डिटेल्स लिखकर एंटर करते हैं।", "footage": "makable", "start": 0.6, "end": 10.1},
    {"text": "साइट का नाम सिलेक्ट करते ही बिल्डिंग प्रोसेस शुरू हो गया।", "footage": "makable", "start": 10.1, "end": 14.0},
    {"text": "सिर्फ छत्तीस सेकंड में makable ने पूरी वेबसाइट तैयार कर दी! लाइव प्रीव्यू में क्रंब एंड को का हीरो सेक्शन, छह बेकरी आइटम्स के साथ उनके प्राइसेस, दुकान की टाइमिंग और फोन नंबर सब कुछ बिल्कुल परफेक्ट दिख रहा है।", "footage": "makable", "start": 18.8, "end": 36.2},
    {"text": "अब इसमें थोड़ा बदलाव करते हैं। हमने चैट में लिखा कि हीरो बैकग्राउंड क्रीम कलर का कर दो और पिकअप बटन जोड़ दो।", "footage": "makable", "start": 54.5, "end": 62.9},
    {"text": "और देखिए, क्रीम बैकग्राउंड के साथ नया बटन भी आ गया! आप भी makable पर अपनी वेबसाइट जरूर बनाइए।", "footage": "makable", "start": 62.9, "end": 69.5}
  ],
  "title": "AI से मिनटों में बेकरी वेबसाइट कैसे बनाएं? makable रिव्यू",
  "description": "देखिए कैसे makable ने सिर्फ एक प्रॉम्प्ट में पूरी बेकरी वेबसाइट तैयार कर दी। इसमें मेन्यू, प्राइसेस और टाइमिंग के साथ लाइव कस्टमाइजेशन भी चुटकियों में हो गया। आप भी makable ट्राई करके अपनी वेबसाइट आसानी से बना सकते हैं।",
  "language": "hi"
}
""",
}
SCRIPT = SCRIPTS[LANGUAGE]  # or a file: SCRIPT = HOME / "runs" / "first-test" / "script.json"

if not run.path("script.json").exists():
    if not isinstance(SCRIPT, Path):
        Path("/content/script.json").write_text(SCRIPT)
        SCRIPT = Path("/content/script.json")
    shorts("import", RUN, "script", SCRIPT)
shorts("new", RUN, "--voice", VOICE_SAMPLE, "--language", LANGUAGE)
script = run.script()
for index, scene in enumerate(script.scenes, 1):
    print(f"{index}. {scene.text}")


## 2. How it's said

Two things to choose by ear before the whole script is spoken: how Chatterbox says makable, and how lively it sounds.

### Saying makable

Chatterbox has no pronunciation setting, so a word it says wrong is respelled in what it reads; the captions keep the script's spelling. This says one sentence with a few spellings. Put the one that sounds right in the run's settings, `shorts("new", RUN, "--pronounce", "makable=…")`, or in `PRONOUNCE` in `shorts/voice.py` for every video.

In [ ]:
WORD = "makable"
SPELLINGS = {  # add your own
    "en": ["makeable", "make-a-bull", "may-kuh-bul", "mayk-uh-bul", "make-able"],
    "hi": ["मेकेबल", "मेकएबल", "मेक-एबल", "मेकबल", "makeable"],
}[LANGUAGE]
SENTENCE = {"en": "Head over and try makable today.", "hi": "आज ही makable ट्राई कीजिए।"}[LANGUAGE]

for spelling in SPELLINGS:
    trial = Voice(VOICE_SAMPLE, LANGUAGE, pronounce={**voice.pronounce, WORD: spelling}, **voice.tts_options)
    print(trial.spoken(SENTENCE))
    display(Audio(str(take(trial, SENTENCE))))


### The delivery

How lively the narration sounds comes from three things:

- **The voice sample.** Chatterbox copies how it's said along with the voice, so a flat sample gives a flat narration. A sample said upbeat and with a smile, not shouted, gives you that. For the "hype" tries, record one said like a Short: fast, without stopping between sentences, your voice going up on the words that matter.
- **Two dials.** `exaggeration` is how much feeling (Chatterbox's default is 0.5; higher is livelier, and faster). `cfg_weight` is how closely it keeps to the sample's pace (default 0.5; lower is slower and looser, which balances a high exaggeration). `shorts.voice.DELIVERY` sets 0.7 and 0.3, Chatterbox's own suggestion for expressive speech. The "hype" tries keep `cfg_weight` at 0.5, since a Short is fast, and go higher on `exaggeration`: too high, and it slurs and drifts from your voice.
- **The words.** Short sentences, and an exclamation mark where something good happens, tend to be said with more lift.

For Hindi from an English voice sample (or any sample in another language), Chatterbox can carry the sample's accent over; its fix is `cfg_weight` 0, the last try.

This speaks one scene a few ways. Put the one you like in the run's settings, `shorts("new", RUN, "--narrator-option", "exaggeration=0.9", "--narrator-option", "cfg_weight=0.3")` (or in `DELIVERY` in `shorts/voice.py`, for every video), then run the narration.

In [ ]:
SCENE = len(script.scenes)  # the scene to try, from 1: the last one, the payoff, is a good test
DELIVERIES = {
    "Chatterbox's default": {"exaggeration": 0.5, "cfg_weight": 0.5},
    "warm": {"exaggeration": 0.6, "cfg_weight": 0.4},
    "lively": {"exaggeration": 0.7, "cfg_weight": 0.3},
    "excited": {"exaggeration": 0.9, "cfg_weight": 0.3},
    "hype": {"exaggeration": 1.0, "cfg_weight": 0.5},  # fast and excited, like a Short
    "hype, more": {"exaggeration": 1.3, "cfg_weight": 0.5},
    "lively, no accent carried over": {"exaggeration": 0.7, "cfg_weight": 0.0},  # a sample in another language
}

text = script.scenes[SCENE - 1].text
print(text)
for name, delivery in DELIVERIES.items():
    wav = take(Voice(VOICE_SAMPLE, LANGUAGE, pronounce=voice.pronounce, **delivery), text)
    print(f"{name}, {delivery}: {duration(wav):.1f} s")
    display(Audio(str(wav)))


## 3. The narration

The `speak` stage, with the run's settings. The first time, Chatterbox downloads its models (a minute or two); then each scene takes a few seconds. Scenes already spoken with the same text, voice sample and settings aren't spoken again.

Under each scene: how long it's spoken and how fast, and for a script written from steps, how long its stretch of the recording is. The video plays the stretch at the speed that fits the scene's narration, so a scene spoken much longer or shorter than its stretch plays it slow or fast. If the speed is far from what the script assumes, change `WORDS_PER_SECOND` in `script.py`.

In [ ]:
shorts("speak", RUN)
script = run.script()
spoken = 0.0
for index, scene in enumerate(script.scenes):
    wav = run.path(narration_file(index))
    seconds = duration(wav)
    spoken += seconds
    stretch = f", over {scene.end - scene.start:.1f} s of {scene.footage}" if scene.start is not None else ""
    print(f"{index + 1}. {seconds:.1f} s, {len(scene.text.split()) / seconds:.1f} words a second{stretch}: {scene.text}")
    display(Audio(str(wav)))
words = sum(len(scene.text.split()) for scene in script.scenes)
print(f"In all: {spoken:.1f} s, {words / spoken:.2f} words a second (the script assumes {WORDS_PER_SECOND:g})")


## 4. The intro in the cloned voice

Only when the voice came from another recording than the intro: if it came from the intro, the intro already has it, so skip this (and set `shorts("new", RUN, "--intro-voice", "as-filmed")`, for the video to use the clip as it is).

The `revoice-intro` stage: it takes the sound out of Omni's clip, separates the speech from the background (Demucs, about a minute), converts the speech to the cloned voice and puts the background back under it, quieter. Below: the clip as Omni made it, then in the cloned voice, to check that the lips still match. A clip that's been done with this voice sample already isn't done again.

In [ ]:
INTRO = HOME / "crumb-and-co" / "runs" / "intro-*.mp4"  # Omni's clip, or a pattern (the newest match); None: the run's

if INTRO is not None:
    clips = sorted(glob.glob(str(INTRO)), key=os.path.getmtime)
    if not clips:
        raise FileNotFoundError(f"Nothing on Drive matches {INTRO}: set INTRO to Omni's clip")
    shorts("import", RUN, "intro", clips[-1])
shorts("revoice-intro", RUN)
print("As Omni made it:")
display(Video(str(run.path(FILMED_INTRO)), embed=True, width=270))
print("In the cloned voice:")
display(Video(str(run.path(VOICED_INTRO)), embed=True, width=270))
